# CADE: correlated neighborhood effect estimation

**Ruiqi Zhang** · UNSW Sydney

CADE combines two-part expression models, nuisance-adjusted source scores and joint sparse support averaging. This notebook is the single implementation for the estimator and its comparison experiments.

Run the definition cells, then the short pilot example. Full experiments are opt-in through `run.py`; no previous reports or saved prediction caches are required. Inputs and predefined protocols are included.


## 1. Estimator and statistical controls


In [ ]:
import hashlib

import warnings

import numpy as np

import pandas as pd

from sklearn.linear_model import LogisticRegression, PoissonRegressor

from sklearn.exceptions import ConvergenceWarning

CADE_ALPHAS = (.01, .1, 1., 10., 100., 1000., 1e6)

CADE_C = 3.841458820694124

def _cd_weights(blocks):
    labels, inverse, counts = np.unique(np.asarray(blocks).astype(str), return_inverse=True, return_counts=True)
    if not len(labels):
        return np.empty(0)
    return len(inverse) / (len(labels) * counts[inverse])

def _cd_score_weights(blocks):
    weights = _cd_weights(blocks)
    return weights / len(weights) if len(weights) else weights

def _cd_fold_ids(blocks):
    labels = sorted(np.unique(np.asarray(blocks).astype(str)), key=lambda v: hashlib.sha256(v.encode()).hexdigest())
    if len(labels) < 3:
        raise ValueError('At least three training blocks required')
    mapping = {label: j % 3 for j, label in enumerate(labels)}
    return np.array([mapping[str(v)] for v in blocks]), mapping

def _cd_bounded(chip):
    index = chip['B2'].index
    values = {}
    for j, channel in enumerate(('clu', 'fasn', 'stk39', 'oligo2')):
        for radius, key in (('near', 'counts_near'), ('far', 'counts_far')):
            n = np.asarray(chip[key], dtype=float)[:, j]
            if np.any(n < 0) or not np.isfinite(n).all():
                raise ValueError('Invalid source counts')
            values[f'source::{channel}::{radius}'] = n / (1. + n)
    for name in ('pooled_annulus::20_50um', 'pooled_annulus::50_100um', 'density::50um', 'density::100um'):
        n = np.expm1(chip['B2'][name].to_numpy(dtype=float))
        if np.any(n < -1e-9) or not np.isfinite(n).all():
            raise ValueError('Invalid log-count input')
        n = np.maximum(n, 0.)
        values[name] = n / (1. + n)
    frame = pd.DataFrame(values, index=index)
    nuisance = [c for c in frame if not c.startswith('source::')]
    sources = [c for c in frame if c.startswith('source::')]
    return frame[nuisance + sources]

def _cd_inverse(matrix):
    try:
        return np.linalg.solve(matrix, np.eye(len(matrix))), False
    except np.linalg.LinAlgError:
        return np.linalg.pinv(matrix, rcond=1e-12), True

def _cd_fit(Z, Y, blocks, alpha, intercept=True):
    Y = np.asarray(Y, dtype=float)
    one = Y.ndim == 1
    if one:
        Y = Y[:, None]
    if not len(Z):
        return np.zeros((Z.shape[1], Y.shape[1])), None, {'n': 0, 'no_selected_training_rows': True}
    w = _cd_weights(blocks)
    penalty = np.eye(Z.shape[1]) * alpha
    if intercept:
        penalty[0, 0] = 0.
    bread, fallback = _cd_inverse(Z.T @ (w[:, None] * Z) + penalty)
    beta = bread @ (Z.T @ (w[:, None] * Y))
    return beta, bread, {'n': len(Z), 'n_design_columns': Z.shape[1], 'blocks': len(np.unique(blocks)), 'alpha': float(alpha),
                         'pseudoinverse_fallback': fallback, 'weights_sum': float(w.sum())}

def _cd_cluster_covariance(Z, y, beta, blocks, bread):
    if not len(Z) or bread is None:
        return np.zeros((Z.shape[1], Z.shape[1])), 0
    w = _cd_weights(blocks)
    residual = y - Z @ beta
    scores = []
    for block in np.unique(blocks):
        ix = blocks == block
        scores.append(Z[ix].T @ (w[ix] * residual[ix]))
    scores = np.asarray(scores)
    return bread @ (scores.T @ scores) @ bread.T, len(scores)

def _cd_extended_glm(design, target, family):
    """Train-only three-block CV for one detection/count response."""
    X = design['Z'][:, 1:]
    T = design['T'][:, 1:]
    losses, warnings_seen, optimization_attempts = np.zeros(len(CADE_ALPHAS)), [], []

    def fit_predict(train, test, alpha):
        y = target[train]
        if family == 'logistic' and len(np.unique(y)) < 2:
            return np.full(len(test), float(y.mean())), None
        if family == 'poisson' and not np.any(y > 0):
            return np.zeros(len(test)), None
        # Uniform numerical policy for every alpha/fold/family: one initial
        # fit, then a fresh deterministic fit with a larger fixed budget.
        # Failed convergence is never silently scored as a weak baseline.
        for attempt, max_iter in enumerate((1000, 5000), 1):
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always', ConvergenceWarning)
                if family == 'logistic':
                    model = LogisticRegression(C=1. / alpha, max_iter=max_iter, tol=1e-7, solver='lbfgs')
                else:
                    # sklearn normalizes by sum weights; alpha/n corresponds
                    # to the declared sum-loss penalty grid.
                    model = PoissonRegressor(alpha=alpha / len(train), max_iter=max_iter, tol=1e-7)
                model.fit(X[train], y, sample_weight=_cd_weights(design['blocks'][train]))
            failed = any(issubclass(w.category, ConvergenceWarning) for w in caught)
            warnings_seen.extend(str(w.message) for w in caught)
            record = {'family': family, 'alpha': float(alpha), 'n_train': len(train),
                      'attempt': attempt, 'max_iter': max_iter,
                      'n_iter': np.asarray(model.n_iter_).astype(int).ravel().tolist(),
                      'converged_without_warning': not failed,
                      'restart': 'fresh deterministic fit, no warm start',
                      'warnings': [str(w.message) for w in caught]}
            optimization_attempts.append(record)
            if not failed:
                break
        if failed:
            error = RuntimeError('Extended baseline did not converge after fixed 1000/5000 budgets: ' + repr(record))
            error.optimization_audit = optimization_attempts
            raise error
        pred = model.predict_proba(test)[:, 1] if family == 'logistic' else model.predict(test)
        return pred, model

    for fold in range(3):
        tr, va = np.flatnonzero(design['folds'] != fold), np.flatnonzero(design['folds'] == fold)
        w = _cd_score_weights(design['blocks'][va]) * len(np.unique(design['blocks'][va])) / len(np.unique(design['blocks']))
        for k, alpha in enumerate(CADE_ALPHAS):
            p, _ = fit_predict(tr, X[va], alpha)
            if family == 'logistic':
                p = np.clip(p, 1e-12, 1. - 1e-12)
                loss = -(target[va] * np.log(p) + (1. - target[va]) * np.log1p(-p))
            else:
                loss = (np.maximum(p, 0.) - target[va]) ** 2
            losses[k] += w @ loss
    best = int(np.argmin(losses))
    pred, model = fit_predict(np.arange(len(target)), T, CADE_ALPHAS[best])
    Tzero = T.copy()
    Tzero[:, design['source'][1:]] = 0.
    if model is None:
        zero_pred = pred.copy()
    else:
        zero_pred = model.predict_proba(Tzero)[:, 1] if family == 'logistic' else model.predict(Tzero)
    integer_target = bool(np.allclose(target, np.rint(target), atol=1e-10, rtol=0))
    return np.maximum(pred, 0.), np.maximum(zero_pred, 0.), {
        'selected_alpha': CADE_ALPHAS[best], 'CV_losses': losses.tolist(),
        'CV_metric': 'binary log loss' if family == 'logistic' else 'clipped count MSE',
        'convergence_warnings': warnings_seen, 'all_warnings_retained': True,
        'optimization_attempts': optimization_attempts,
        'numerical_retry_policy': '1000 then 5000 fresh deterministic iterations; convergence warning after retry raises; no scored nonconverged fit',
        'response_integer_valued': integer_target,
        'Poisson_objective_interpretation': ('count likelihood working model' if integer_target else
            'quasi-likelihood conditional-mean objective for nonnegative continuous outcomes; not an exact count likelihood') if family == 'poisson' else None,
        'features': 'same unscaled bounded B3; no raw coordinates',
        'source_zero_contrast': 'model contrast from setting the eight bounded source features to zero; not a causal intervention'}

from pathlib import Path

import json,hashlib,time

import numpy as np

import pandas as pd

from scipy.special import expit

from scipy import sparse

from threadpoolctl import threadpool_limits

def ca_rng(*parts):
    seed=np.frombuffer(hashlib.sha256('|'.join(map(str,parts)).encode()).digest(),dtype='<u4')
    return np.random.default_rng(np.random.SeedSequence(seed))

def ca_weights(blocks,keep=None):
    b=np.asarray(blocks).astype(str)
    if keep is not None:b=b[np.asarray(keep)]
    _,ix,n=np.unique(b,return_inverse=True,return_counts=True)
    return 1/(len(n)*n[ix])

def ca_fields(chip):
    near=np.asarray(chip['counts_near'],float);S=near/(1+near)
    pool=np.expm1(chip['B2']['pooled_annulus::20_50um'].to_numpy())
    density=np.expm1(chip['B2']['density::50um'].to_numpy())
    base=2+.2*pool/(1+pool)+.1*density/(1+density)
    return S,base[:,None]+np.zeros_like(S)

def ca_synthetic(chip,p,alpha,rep,protocol,mechanism='independent_mask'):
    S,base=ca_fields(chip);mu=base+alpha*S
    rng=ca_rng(protocol['seed'],mechanism,p,alpha,rep,chip['chip_id'])
    constant_capture=None
    if mechanism in ['independent_mask','chip_background_shift']:
        if mechanism=='chip_background_shift' and chip['chip_id']=='A03599E2':mu=mu+.8
        latent=mu+rng.uniform(-.25,.25,mu.shape);prob=np.full_like(mu,p)
        y=latent*(rng.random(mu.shape)<prob);obs=prob*mu;pos=mu.copy();shift=alpha*S
        constant_capture=p
    elif mechanism in ['poisson_ordinary','poisson_thinning']:
        capture=1. if mechanism=='poisson_ordinary' else .1
        mu=(.05 if capture==1 else .5)*(1+3*S)
        obs=capture*mu;y=rng.poisson(obs).astype(float)
        prob=-np.expm1(-obs);pos=obs/prob
        shift=(.15 if capture==1 else 1.5)*S;constant_capture=capture
    elif mechanism=='occurrence_only':
        mu=.03+.09*S;prob=mu.copy();obs=mu.copy();pos=np.ones_like(mu)
        y=(rng.random(mu.shape)<mu).astype(float);shift=.09*S;constant_capture=1.
    elif mechanism=='capture_depends_exposure':
        prob=.03+.30*S;latent=mu+rng.uniform(-.25,.25,mu.shape)
        y=latent*(rng.random(mu.shape)<prob);obs=prob*mu;pos=mu.copy();shift=alpha*S
    elif mechanism=='capture_depends_intensity':
        latent=mu+rng.uniform(-.25,.25,mu.shape)
        # Selection depends on unobserved U even after conditioning on X.
        zprob=expit(np.log(.08/.92)+8*(latent-2.15))
        y=latent*(rng.random(mu.shape)<zprob)
        nodes,weights=np.polynomial.legendre.leggauss(32)
        values=mu[:,:,None]+.25*nodes
        rates=expit(np.log(.08/.92)+8*(values-2.15))
        prob=np.sum(rates*weights,axis=2)/2
        obs=np.sum(values*rates*weights,axis=2)/2
        pos=obs/prob;shift=alpha*S
    else:raise ValueError(mechanism)
    assert np.isfinite(y).all() and np.all(mu>0) and np.all(prob>0)
    return {'Y':y,'latent_mean':mu,'observed_mean':obs,'positive_mean':pos,
            'latent_shift':shift,'prob_positive':prob,'constant_capture':constant_capture,
            'Y_sha256':hashlib.sha256(y.astype('<f8').tobytes()).hexdigest()}

from scipy.special import expit, logit

from scipy.optimize import minimize_scalar

import time

def _ef_cv_ids(blocks):
    labels = sorted(np.unique(np.asarray(blocks).astype(str)), key=lambda v: hashlib.sha256(v.encode()).hexdigest())
    k = min(3, len(labels))
    if k < 2:
        return np.zeros(len(blocks), int), 1
    mapping = {label: j % k for j, label in enumerate(labels)}
    return np.array([mapping[str(v)] for v in blocks]), k

def _ef_logit_fit(Z, y, blocks, alpha, offset=None, initial=None):
    """Convex weighted binomial loss + alpha/2 * ||nonintercept||^2.

    Newton steps with Armijo backtracking; gradients use the sum-loss scale,
    matching LogisticRegression C=1/alpha with the same sample weights.
    """
    n, p = Z.shape
    if offset is None:
        offset = np.zeros(n)
    w = _cd_weights(blocks)
    if not n:
        return np.zeros(p), np.zeros((p,p)), {'constant': 0., 'iterations': 0, 'gradient_max': 0., 'converged': True}
    q = float(w @ y / w.sum())
    if np.all(y == y[0]):
        beta = np.zeros(p)
        beta[0] = logit(np.clip(q, 1e-12, 1 - 1e-12))
        return beta, np.zeros((p,p)), {'constant': q, 'iterations': 0, 'gradient_max': 0., 'converged': True}
    penalty = np.full(p, float(alpha)); penalty[0] = 0.
    beta = np.zeros(p) if initial is None else initial.copy()
    if initial is None:
        beta[0] = logit(np.clip(q, 1e-8, 1-1e-8)) - float(w @ offset / w.sum())
    def obj(b):
        eta = Z @ b + offset
        return float(w @ (np.logaddexp(0., eta) - y*eta) + .5*np.dot(penalty*b,b))
    value = obj(beta)
    converged = False
    for iteration in range(1, 101):
        eta = Z @ beta + offset
        prob = expit(eta)
        grad = Z.T @ (w*(prob-y)) + penalty*beta
        hess = Z.T @ ((w*prob*(1-prob))[:,None]*Z) + np.diag(penalty)
        if float(np.max(np.abs(grad))) <= 2e-8 * max(1.,w.sum()):
            converged = True
            break
        try:
            step = np.linalg.solve(hess, grad)
        except np.linalg.LinAlgError:
            step = np.linalg.pinv(hess, rcond=1e-12) @ grad
        descent = float(grad @ step)
        fraction = 1.
        for backtrack in range(35):
            proposal = beta-fraction*step
            val = obj(proposal)
            if val <= value - 1e-4*fraction*descent + 1e-12:
                break
            fraction *= .5
        else:
            raise FloatingPointError('Logistic Newton backtracking failed')
        beta, value = proposal, val
    eta = Z @ beta + offset
    prob = expit(eta)
    grad = Z.T @ (w*(prob-y)) + penalty*beta
    hess = Z.T @ ((w*prob*(1-prob))[:,None]*Z) + np.diag(penalty)
    gradient_max = float(np.max(np.abs(grad)))
    converged = converged or gradient_max <= 2e-8*max(1.,w.sum())
    if not converged:
        raise FloatingPointError('Logistic Newton did not converge: '+str(gradient_max))
    bread, fallback = _cd_inverse(hess)
    return beta, bread, {'constant': None, 'iterations': iteration, 'gradient_max': gradient_max,
        'scaled_gradient_max': gradient_max/max(1.,w.sum()), 'converged': converged,
        'inverse_fallback': fallback, 'objective': value}

def _ef_fit(Z, y, blocks, alpha, family, covariance=False, initial=None):
    if family == 'positive':
        beta, bread, audit = _cd_fit(Z, y, blocks, alpha)
        beta = beta.ravel()
        audit['constant'] = 0. if not len(Z) else None
    else:
        beta, bread, audit = _ef_logit_fit(Z,y,blocks,alpha,initial=initial)
    result = {'beta': beta, 'alpha': float(alpha), 'family': family, 'constant': audit.get('constant'), 'audit': audit}
    if covariance:
        if family == 'positive':
            cov, nblocks = _cd_cluster_covariance(Z,y,beta,blocks,bread)
        elif result['constant'] is not None:
            cov, nblocks = np.zeros((Z.shape[1],Z.shape[1])), len(np.unique(blocks))
        else:
            w = _cd_weights(blocks);residual=y-expit(Z@beta)
            _,inverse = np.unique(blocks, return_inverse=True)
            scores=np.zeros((len(np.unique(blocks)), Z.shape[1]))
            np.add.at(scores,inverse,Z*(w*residual)[:,None])
            cov=bread@(scores.T@scores)@bread.T;nblocks=len(scores)
        result['variance']=np.maximum(np.diag(cov),0.)
        result['nblocks']=nblocks
        result['support'] = np.ptp(Z,axis=0)>1e-12 if len(Z) else np.zeros(Z.shape[1],bool)
    return result

def _ef_predict(fit,Z):
    if fit.get('constant') is not None:
        return np.full(len(Z),fit['constant'])
    raw=Z@fit['beta']
    return expit(raw) if fit['family']=='detection' else np.maximum(raw,0.)

def _ef_alpha(Z,y,blocks,family):
    folds,kfold=_ef_cv_ids(blocks)
    losses=np.zeros(len(CADE_ALPHAS))
    if not len(y) or kfold<2:
        return float(CADE_ALPHAS[-1]), {'CV_losses': losses.tolist(), 'folds': kfold, 'insufficient_blocks': True}
    allblocks=len(np.unique(blocks))
    for fold in range(kfold):
        tr=folds!=fold;va=~tr
        ww=_cd_score_weights(blocks[va])*len(np.unique(blocks[va]))/allblocks
        initial=None
        # A decreasing-penalty path changes optimization only, not the grid.
        for j in range(len(CADE_ALPHAS)-1,-1,-1):
            fit=_ef_fit(Z[tr],y[tr],blocks[tr],CADE_ALPHAS[j],family,initial=initial)
            initial=fit['beta'] if fit['constant'] is None else None
            pred=_ef_predict(fit,Z[va])
            if family=='detection':
                pred=np.clip(pred,1e-12,1-1e-12)
                loss=-(y[va]*np.log(pred)+(1-y[va])*np.log1p(-pred))
            else:
                loss=(pred-y[va])**2
            losses[j]+=float(ww@loss)
    chosen=int(np.argmin(losses))
    return float(CADE_ALPHAS[chosen]), {'CV_losses':losses.tolist(),'folds':kfold,'selected_alpha':float(CADE_ALPHAS[chosen]),
        'metric':'binary_logloss' if family=='detection' else 'positive_MSE'}

def _ef_eb_factors(beta,variance,support):
    """Zero-centered heteroscedastic working-normal marginal likelihood.

    Estimate tau^2 >= 0 by a deterministic log-grid plus local refinement,
    then posterior-mean multiplier tau^2/(tau^2+v). Inactive design columns
    are excluded from hyperparameter fitting and receive zero multiplier.
    """
    out=np.zeros(len(beta));keep=np.asarray(support)&np.isfinite(beta)&np.isfinite(variance)
    if not keep.any():
        return out,{'tau2':0.,'n_supported':0,'marginal_objective':None}
    b=np.asarray(beta)[keep];v=np.maximum(np.asarray(variance)[keep],1e-12)
    scale=max(float(np.max(b*b)),float(np.max(v)),1e-12)
    def nll(t):
        return .5*float(np.sum(np.log(v+t)+b*b/(v+t)))
    loggrid=np.linspace(np.log(scale)-25,np.log(scale)+3,81)
    tg=np.r_[0.,np.exp(loggrid)]
    values=np.array([nll(t) for t in tg]);best=int(np.argmin(values))
    tau=float(tg[best]);bestval=float(values[best])
    if best>0:
        lo=loggrid[max(0,best-2)];hi=loggrid[min(len(loggrid)-1,best)]
        if hi>lo:
            result=minimize_scalar(lambda l:nll(np.exp(l)),bounds=(lo,hi),method='bounded',options={'xatol':1e-8})
            if result.success and result.fun<bestval:
                tau=float(np.exp(result.x));bestval=float(result.fun)
    out[keep]=tau/(tau+v)
    return out,{'tau2':tau,'n_supported':int(keep.sum()),'marginal_objective':bestval,
        'variance_floor':1e-12,'prior':'independent working N(0,tau2) on source coefficients',
        'hyperparameter_information':'training coefficients and CR0 variances only'}

def _ef_shrink(base,Z,y,blocks,source,kind,c=None):
    out={**base,'beta':base['beta'].copy()}
    before=base['beta'][source].copy();variance=base['variance'][source]
    support=base['support'][source]
    extra={}
    if kind=='plain' or (kind=='CV' and c==0.):
        factors=np.ones(len(before))
    elif kind=='EB':
        factors,extra=_ef_eb_factors(before,variance,support)
    else:
        cutoff=CADE_C if kind=='fixed' else float(c)
        if np.isinf(cutoff):
            factors=np.zeros(len(before))
        else:
            ratio=np.divide(cutoff*variance,before*before,out=np.full(len(before),np.inf),where=before!=0)
            factors=np.maximum(0.,1-ratio)
        factors[~support]=0.
    if kind!='plain' and not (kind=='CV' and c==0.) and base['nblocks']<2:
        factors[:]=0.
    out['beta'][source]*=factors
    nuisance=~source
    # Exact c=0/no-shrink identity; all other fits condition on fixed sources.
    needs_refit=not np.array_equal(out['beta'][source],base['beta'][source])
    refit={}
    if needs_refit and len(y) and base.get('constant') is None:
        offset=Z[:,source]@out['beta'][source]
        if base['family']=='positive':
            nb,_,refit=_cd_fit(Z[:,nuisance],y-offset,blocks,base['alpha'])
            out['beta'][nuisance]=nb.ravel()
        else:
            nb,_,refit=_ef_logit_fit(Z[:,nuisance],y,blocks,base['alpha'],offset=offset,initial=base['beta'][nuisance])
            out['beta'][nuisance]=nb
    out['shrink_audit']={'kind':kind,'c':('inf' if c is not None and np.isinf(c) else c),
        'source_coefficients_before':before.tolist(),'source_variances':variance.tolist(),
        'source_support':support.tolist(),'factors':factors.tolist(),'source_coefficients_after':out['beta'][source].tolist(),
        'positive_or_detection_clusters':int(base['nblocks']),'nuisance_refit':needs_refit,'refit_audit':refit,**extra}
    return out

from pathlib import Path

import json, hashlib, time, gzip

import numpy as np

import pandas as pd

from scipy.special import expit

from threadpoolctl import threadpool_limits

def ef_truth(chip, data, p, alpha, mechanism):
    S, base = ca_fields(chip)
    if mechanism=='chip_background_shift' and chip['chip_id']=='A03599E2':
        base=base+.8
    if mechanism in ['independent_mask','chip_background_shift']:
        zero=p*base
        variance=p*(.25**2/3)+p*(1-p)*data['latent_mean']**2
    elif mechanism in ['poisson_ordinary','poisson_thinning']:
        zero=np.full_like(base,.05);variance=data['observed_mean']
    elif mechanism=='occurrence_only':
        zero=np.full_like(base,.03)
        variance=data['observed_mean']*(1-data['observed_mean'])
    elif mechanism=='capture_depends_exposure':
        zero=.03*base
        q=data['prob_positive']; mu=data['latent_mean']
        variance=q*(.25**2/3)+q*(1-q)*mu**2
    elif mechanism=='capture_depends_intensity':
        nodes,w=np.polynomial.legendre.leggauss(32)
        v=base[:,:,None]+.25*nodes
        q=expit(np.log(.08/.92)+8*(v-2.15))
        zero=np.sum(v*q*w,axis=2)/2
        v=data['latent_mean'][:,:,None]+.25*nodes
        q=expit(np.log(.08/.92)+8*(v-2.15))
        variance=np.sum(v*v*q*w,axis=2)/2-data['observed_mean']**2
    else: raise ValueError(mechanism)
    return {'observed_zero':zero,'observed_effect':data['observed_mean']-zero,
        'variance':variance,'latent_effect':alpha*S,
        'positive_equals_latent':mechanism in ['independent_mask','chip_background_shift']}

def ef_block_mse(blocks, a, b):
    return float(np.mean([np.mean((a[np.asarray(blocks)==v].mean(0)-b[np.asarray(blocks)==v].mean(0))**2)
                          for v in np.unique(blocks)]))

def ef_score(chip, data, truth, fit):
    pred=fit['observed_prediction']; zero=fit['observed_zero_prediction']
    delta=pred-zero; w=ca_weights(chip['blocks'])
    mse=lambda a,b:float(np.mean(w@((a-b)**2)))
    t=truth['observed_effect']; cosine=[];sign=[]
    for g in range(t.shape[1]):
        norm=np.sqrt((w@t[:,g]**2)*(w@delta[:,g]**2))
        cosine.append(float(w@(t[:,g]*delta[:,g])/norm) if norm>1e-20 else np.nan)
        keep=np.abs(t[:,g])>1e-12
        sign.append(float(w[keep]@(np.sign(delta[keep,g])==np.sign(t[keep,g]))/w[keep].sum()) if keep.any() else np.nan)
    row={'observed_mean_MSE':mse(data['observed_mean'],pred),
        'observed_effect_MSE':mse(t,delta),'observed_MSE':mse(data['Y'],pred),
        'block_mean_MSE':ef_block_mse(chip['blocks'],data['observed_mean'],pred),
        'block_observed_MSE':ef_block_mse(chip['blocks'],data['Y'],pred),
        'oracle_observed_MSE':mse(data['Y'],data['observed_mean']),
        'irreducible_variance':float(np.mean(w@truth['variance'])),
        'observed_effect_cosine':float(np.nanmean(cosine)) if np.isfinite(cosine).any() else np.nan,
        'direction_accuracy':float(np.nanmean(sign)) if np.isfinite(sign).any() else np.nan,
        'positive_branch_latent_shift_MSE':np.nan,'oracle_scaled_latent_effect_MSE':np.nan}
    if truth['positive_equals_latent'] and fit.get('positive_prediction') is not None:
        row['positive_branch_latent_shift_MSE']=mse(truth['latent_effect'],fit['positive_prediction']-fit['positive_zero_prediction'])
    cap=data['constant_capture']
    if cap is not None:
        row['oracle_scaled_latent_effect_MSE']=mse(t/cap,delta/cap)
    row['expected_observed_risk']=row['observed_mean_MSE']+row['irreducible_variance']
    return row

from scipy.special import logsumexp, ndtri

from scipy.optimize import minimize

from scipy.stats import qmc

def _js_if(Z,y,blocks,base,allblocks):
    """Aligned block influence scores using the original branch weights."""
    p=Z.shape[1];out=np.zeros((len(allblocks),p))
    if not len(Z) or base['constant'] is not None:return out
    w=_cd_weights(blocks);penalty=np.eye(p)*base['alpha'];penalty[0,0]=0.
    if base['family']=='detection':
        prob=expit(Z@base['beta']);res=y-prob
        H=Z.T@((w*prob*(1-prob))[:,None]*Z)+penalty
    else:
        res=y-Z@base['beta'];H=Z.T@(w[:,None]*Z)+penalty
    bread,_=_cd_inverse(H)
    lookup={b:i for i,b in enumerate(allblocks)}
    scores=np.zeros_like(out)
    np.add.at(scores,np.array([lookup[b] for b in blocks]),Z*(w*res)[:,None])
    return scores@bread.T

def _js_base_pack(Z,Y,blocks,source):
    packs=[];summaries=[];covariances=[];support=[]
    labels=np.unique(blocks)
    for g in range(Y.shape[1]):
        keep=Y[:,g]>0
        qa,qaa=_ef_alpha(Z,keep.astype(float),blocks,'detection')
        ma,maa=_ef_alpha(Z[keep],Y[keep,g],blocks[keep],'positive')
        qb=_ef_fit(Z,keep.astype(float),blocks,qa,'detection',covariance=True)
        mb=_ef_fit(Z[keep],Y[keep,g],blocks[keep],ma,'positive',covariance=True)
        iq=_js_if(Z,keep.astype(float),blocks,qb,labels)
        im=_js_if(Z[keep],Y[keep,g],blocks[keep],mb,labels)
        pair=np.stack([iq[:,source],im[:,source]],axis=-1)
        cov=np.einsum('bsi,bsj->sij',pair,pair)
        # This equality ensures the stronger prior receives the same diagonals.
        assert np.allclose(cov[:,0,0],qb['variance'][source],rtol=1e-8,atol=1e-12)
        assert np.allclose(cov[:,1,1],mb['variance'][source],rtol=1e-8,atol=1e-12)
        packs.append({'q':qb,'m':mb,'keep':keep,'q_alpha':qaa,'m_alpha':maa})
        summaries.append(np.column_stack([qb['beta'][source],mb['beta'][source]]))
        covariances.append(cov)
        support.append(np.column_stack([qb['support'][source] & (qb['constant'] is None),mb['support'][source] & (mb['constant'] is None)]))
    return packs,np.concatenate(summaries),np.concatenate(covariances),np.concatenate(support)

def _js_uni_prior(b,v,supported):
    """Point normal marginal likelihood on a deterministic data-scaled grid."""
    supported=np.asarray(supported,bool);N=len(b)
    mean=np.zeros(N);variance=np.zeros(N);inclusion=np.zeros(N)
    k=np.flatnonzero(supported)
    if not len(k):return mean,variance,inclusion,{'tau2':0.,'pi_slab':0.,'n_supported':0}
    y=np.asarray(b)[k];vv=np.maximum(np.asarray(v)[k],1e-12)
    scale=max(float(np.max(y*y)),float(np.median(vv)),1e-12)
    taus=scale*np.geomspace(1e-3,4.,17)
    ll0=-.5*(np.log(2*np.pi*vv)+y*y/vv)
    ll1=-.5*(np.log(2*np.pi*(vv[None,:]+taus[:,None]))+y[None,:]**2/(vv[None,:]+taus[:,None]))
    odds=ll1-ll0
    lo=np.full(len(taus),1e-12);hi=np.full(len(taus),1-1e-12)
    # Concavity in pi makes this vectorized derivative bisection global.
    for _ in range(48):
        mid=(lo+hi)/2
        r=expit(np.log(mid/(1-mid))[:,None]+odds)
        derivative=np.sum(r/mid[:,None]-(1-r)/(1-mid)[:,None],axis=1)
        lo=np.where(derivative>0,mid,lo);hi=np.where(derivative>0,hi,mid)
    pi=(lo+hi)/2
    val=logsumexp(np.stack([np.log1p(-pi)[:,None]+np.broadcast_to(ll0,ll1.shape),np.log(pi)[:,None]+ll1]),axis=0).sum(1)
    null=float(ll0.sum());full=ll1.sum(1)
    val0=val.copy();pi=np.where(full>val,1.,pi);val=np.maximum(val,full)
    pi=np.where(null>=val,0.,pi);val=np.maximum(val,null)
    chosen=int(np.argmax(val));tau=float(taus[chosen]);pp=float(pi[chosen])
    if pp==0.:
        tau=0.;r=np.zeros(len(k))
    elif pp==1.:r=np.ones(len(k))
    else:r=expit(np.log(pp/(1-pp))+odds[chosen])
    a=tau/(tau+vv) if tau else np.zeros_like(vv)
    cm=a*y;cv=a*vv
    mean[k]=r*cm;variance[k]=np.maximum(0.,r*(cv+cm*cm)-mean[k]**2);inclusion[k]=r
    return mean,variance,inclusion,{'tau2':tau,'pi_slab':pp,'n_supported':len(k),
        'log_marginal_likelihood':float(val[chosen]),'tau_grid':taus.tolist(),
        'grid_pi':pi.tolist(),'grid_loglik':val.tolist(),'variance_floor':1e-12,
        'scale_grid_rule':'max(max supported coefficient squared,median CR0 variance)*geomspace(.001,4,17)',
        'pi_optimization':'48-step concave derivative bisection plus exact0/1 endpoints',
        'pooled_units':N}

def _js_refit(base,Z,y,blocks,source,coeff):
    out={**base,'beta':base['beta'].copy()};out['beta'][source]=coeff
    if len(y) and base['constant'] is None:
        n=~source;offset=Z[:,source]@coeff
        if base['family']=='detection':
            beta,_,audit=_ef_logit_fit(Z[:,n],y,blocks,base['alpha'],offset=offset,initial=base['beta'][n])
        else:
            beta,_,audit=_cd_fit(Z[:,n],y-offset,blocks,base['alpha']);beta=beta.ravel()
        out['beta'][n]=beta
    else:audit={'constant_branch':base['constant']}
    out['refit_audit']=audit
    return out

def _js_prediction(qfit,mfit,T,source):
    T0=T.copy();T0[:,source]=0.
    q=_ef_predict(qfit,T);q0=_ef_predict(qfit,T0)
    m=_ef_predict(mfit,T);m0=_ef_predict(mfit,T0)
    return {'observed_prediction':q*m,'observed_zero_prediction':q0*m0,
            'positive_prediction':m,'positive_zero_prediction':m0,
            'detection_prediction':q,'detection_zero_prediction':q0}

from pathlib import Path

import json,hashlib,time,gzip

import numpy as np

import pandas as pd

from scipy.special import expit,logit

from threadpoolctl import threadpool_limits

JS_EXTRA_STRESSES=['signed_sparse','mixed_sources','opposing_branches']

def js_generate(chip,p,alpha,rep,protocol,mechanism):
    if mechanism not in JS_EXTRA_STRESSES:
        data=ca_synthetic(chip,p,alpha,rep,protocol,mechanism)
        truth=ef_truth(chip,data,p,alpha,mechanism)
        return data,truth
    S,base=ca_fields(chip);F=np.asarray(chip['counts_far'],float);F=F/(1+F)
    if mechanism=='signed_sparse':shift=-.75*S
    elif mechanism=='mixed_sources':shift=.6*S-.4*np.roll(S,1,axis=1)+.35*F
    else:shift=-.6*S
    mu=base+shift; assert np.all(mu>.25)
    q=expit(logit(p)+2*S) if mechanism=='opposing_branches' else np.full_like(mu,p)
    rng=ca_rng(protocol['seed'],mechanism,p,alpha,rep,chip['chip_id'])
    U=mu+rng.uniform(-.25,.25,mu.shape);Y=U*(rng.random(mu.shape)<q)
    mean=q*mu;zero=p*base
    data={'Y':Y,'latent_mean':mu,'observed_mean':mean,'positive_mean':mu,
          'latent_shift':shift,'prob_positive':q,'constant_capture':None if mechanism=='opposing_branches' else p,
          'Y_sha256':hashlib.sha256(Y.astype('<f8').tobytes()).hexdigest()}
    truth={'observed_zero':zero,'observed_effect':mean-zero,'variance':q*.25**2/3+q*(1-q)*mu**2,
           'latent_effect':shift,'positive_equals_latent':True}
    return data,truth

def userpip_coefficients(beta, variance, supported):
    """Apply the exact requested pi/tau/PIP formula on supported coordinates."""
    beta=np.asarray(beta,float);variance=np.asarray(variance,float)
    supported=np.asarray(supported,bool)
    assert beta.shape==variance.shape==supported.shape and len(beta)==8
    assert np.isfinite(beta).all() and np.isfinite(variance).all()
    assert np.min(variance)>=-1e-12
    v=np.maximum(variance,0.);ix=np.flatnonzero(supported)
    pip=np.zeros(8);slab=np.zeros(8)
    pi=1./8.;floor=1e-12
    if len(ix):
        tau2=max(float(np.max(beta[ix]**2-v[ix])),float(np.mean(v[ix])))
        if tau2>0:
            vv=np.maximum(v[ix],floor)
            logodds=np.log(pi/(1-pi))-.5*np.log1p(tau2/vv)+.5*beta[ix]**2*tau2/(vv*(vv+tau2))
            pip[ix]=expit(logodds)
            slab[ix]=tau2/(tau2+vv)
        else:
            # Both components coincide at zero: inclusion remains the prior,
            # but every posterior source coefficient is exactly zero.
            pip[ix]=pi
    else:tau2=0.
    factors={
        'userpip_posterior_mean':pip*slab,
        'userpip_hard_02':(pip>=.2).astype(float)*slab,
        'userpip_soft_02':np.maximum(0.,(pip-.2)/.8)*slab,
        'userpip_soft_05':np.maximum(0.,(pip-.5)/.5)*slab,
    }
    coefficients={name:beta*weight for name,weight in factors.items()}
    return coefficients,{'pi_slab':pi,'tau2':tau2,'n_supported':int(supported.sum()),
        'source_coefficients_before':beta.tolist(),'source_variances':variance.tolist(),
        'source_support':supported.tolist(),'posterior_inclusion_probability':pip.tolist(),
        'conditional_slab_multiplier':slab.tolist(),
        'decision_multipliers':{k:x.tolist() for k,x in factors.items()},
        'likelihood_variance_floor':floor,'n_variance_floored':int(np.sum(supported&(v<floor))),
        'tau_rule':'max(max_supported(beta²−raw_CR0variance),mean_supported(raw_CR0variance))',
        'pooling':'8 source coordinates within one response and one branch; unsupported excluded; pi stays1/8'}

from scipy.stats import chi2

BA_FIELDS=['observed_prediction','observed_zero_prediction','positive_prediction',
    'positive_zero_prediction','detection_prediction','detection_zero_prediction']

import hashlib

import numpy as np

from scipy.special import expit, logit, logsumexp

from scipy.optimize import minimize_scalar

GM_SCORE_STATES=((np.arange(256)[:,None]>>np.arange(8))&1).astype(float)

def _gm_weights(blocks):
    _,inverse,count=np.unique(np.asarray(blocks).astype(str),return_inverse=True,return_counts=True)
    return len(inverse)/(len(count)*count[inverse]) if len(count) else np.empty(0)

def _gm_pinv(matrix,relative=1e-10):
    matrix=(matrix+matrix.T)/2;e,U=np.linalg.eigh(matrix)
    cutoff=max(float(np.max(e)),1e-30)*relative;keep=e>cutoff
    inv=(U[:,keep]/e[keep])@U[:,keep].T if keep.any() else np.zeros_like(matrix)
    return inv,{'rank':int(keep.sum()),'dimension':len(e),'eigenvalues':e.tolist(),
        'relative_rank_cutoff':relative}

def _gm_nuisance(N,y,blocks,offset,family):
    n,p=N.shape;w=_gm_weights(blocks)
    if not n:return {'beta':np.zeros(p),'constant':0.,'converged':True,'rank':0,'iterations':0}
    if family=='positive':
        inverse,audit=_gm_pinv(N.T@(w[:,None]*N))
        beta=inverse@(N.T@(w*(y-offset)))
        return {'beta':beta,'constant':None,'converged':True,'rank':audit['rank'],
            'rank_audit':audit,'iterations':1}
    if np.all(y==y[0]):
        beta=np.zeros(p);beta[0]=logit(np.clip(y[0],1e-12,1-1e-12))
        return {'beta':beta,'constant':float(y[0]),'converged':True,'rank':0,'iterations':0}
    beta=np.zeros(p);beta[0]=logit(np.clip(w@y/w.sum(),1e-10,1-1e-10))-w@offset/w.sum()
    objective=lambda b:float(w@(np.logaddexp(0.,N@b+offset)-y*(N@b+offset)))
    value=objective(beta);converged=False
    for it in range(100):
        q=expit(N@beta+offset);gradient=N.T@(w*(q-y))
        H=N.T@((w*q*(1-q))[:,None]*N);inverse,audit=_gm_pinv(H)
        if np.max(abs(gradient))<2e-8*max(w.sum(),1.):converged=True;break
        step=inverse@gradient;decrease=float(gradient@step);fraction=1.
        for _ in range(40):
            proposed=beta-fraction*step;nextvalue=objective(proposed)
            if nextvalue<=value-1e-4*fraction*decrease+1e-12:break
            fraction*=.5
        beta,value=proposed,nextvalue
    q=expit(N@beta+offset);gradient=N.T@(w*(q-y))
    converged=converged or np.max(abs(gradient))<2e-8*max(w.sum(),1.)
    if not converged:raise FloatingPointError('Unpenalized nuisance logistic fit did not converge')
    return {'beta':beta,'constant':None,'converged':True,'rank':audit['rank'],
        'rank_audit':audit,'iterations':it+1,'scaled_gradient_max':float(np.max(abs(gradient))/w.sum())}

def _gm_noise_scale(N,S,y):
    """Positive-branch homoskedastic working scale from unpenalized full OLS."""
    if not len(y):return 1.,{'rank':0,'df':0,'fallback':True}
    Z=np.column_stack([N,S]);b,_,rank,_=np.linalg.lstsq(Z,y,rcond=1e-10)
    df=max(len(y)-rank,1);raw=float(np.sum((y-Z@b)**2)/df)
    floor=max(1e-12,1e-10*float(np.mean(y*y)))
    return max(raw,floor),{'rank':int(rank),'df':int(df),'raw_sigma2':raw,
        'floor':floor,'scale_model':'Unweighted full-linear residual SSE/(n-rank); working homoskedasticity, no outcome scale cross-validation.'}

def _gm_profile(N,S,y,blocks,beta_source,family,allblocks):
    fit=_gm_nuisance(N,y,blocks,S@beta_source,family);w=_gm_weights(blocks)
    if not len(y) or fit['constant'] is not None:
        return {'score_U':np.zeros(8),'pseudo_score':np.zeros(8),'information_I':np.zeros((8,8)),
            'covariance_model':np.zeros((8,8)),'covariance_cluster':np.zeros((8,8)),
            'aligned_block_scores':np.zeros((len(allblocks),8)),'source_coefficients':beta_source,
            'nuisance_fit':fit,'source_rank':0,'n':len(y),'family':family,'blocks':list(allblocks)}
    raw=N@fit['beta']+S@beta_source
    if family=='detection':
        mean=expit(raw);variance=mean*(1-mean);h=w*variance;scale_audit={}
    else:
        mean=raw;scale,scale_audit=_gm_noise_scale(N,S,y);variance=np.full(len(y),scale);h=w
    Hnn=N.T@(h[:,None]*N);inverse,nuisance_rank=_gm_pinv(Hnn)
    R=S-N@(inverse@(N.T@(h[:,None]*S)))
    I=R.T@(h[:,None]*R);I=(I+I.T)/2
    individual=R*(w*(y-mean))[:,None];U=individual.sum(0)
    lookup={label:j for j,label in enumerate(allblocks)};scores=np.zeros((len(allblocks),8))
    np.add.at(scores,np.array([lookup[v] for v in blocks]),individual)
    Vcluster=scores.T@scores;Vmodel=R.T@((w*w*variance)[:,None]*R)
    _,rank_audit=_gm_pinv(I)
    return {'score_U':U,'pseudo_score':I@beta_source+U,'information_I':I,
        'covariance_model':Vmodel,'covariance_cluster':Vcluster,'aligned_block_scores':scores,
        'source_coefficients':beta_source.copy(),'nuisance_fit':fit,'source_rank':rank_audit['rank'],
        'information_rank_audit':rank_audit,'nuisance_information_rank':nuisance_rank,
        'nuisance_orthogonality_max_error':float(np.max(abs(N.T@(h[:,None]*R)))),
        'noise_scale':scale_audit,'n':len(y),'family':family,'blocks':list(allblocks),
        'cluster_meat_note':'Raw residual score meat; under misspecified/null starting means it may contain between-block signal, not an unbiased variance claim.'}

def _gm_likelihood_group(summary,covariance):
    I=summary['information_I'];V=summary['covariance_'+covariance];u=summary['pseudo_score']
    e,Q=np.linalg.eigh(I);cut=max(float(e.max()),1e-30)*1e-10;keep=e>cut
    if not keep.any():return None
    Q=Q[:,keep];A=Q.T@I;observed=Q.T@u;C=Q.T@V@Q
    C=(C+C.T)/2;ce,U=np.linalg.eigh(C)
    floor=max(float(ce.max())*1e-8,1e-12)
    C=(U*np.maximum(ce,floor))@U.T
    return {'A':A,'u':observed,'V':C,'rank':int(keep.sum()),
        'covariance_eigenvalues_before_floor':ce.tolist(),'covariance_eigenvalue_floor':floor}

def _gm_support_components(group,tau2):
    if group is None:return np.zeros(256),np.zeros((256,8))
    A,u,V=group['A'],group['u'],group['V']
    gram=np.einsum('ik,sk,jk->sij',A,GM_SCORE_STATES,A)
    C=V[None,:,:]+tau2*gram
    solved=np.linalg.solve(C,np.broadcast_to(u,(256,len(u)))[...,None])[...,0]
    logL=-.5*(len(u)*np.log(2*np.pi)+np.linalg.slogdet(C)[1]+solved@u)
    means=tau2*GM_SCORE_STATES*(solved@A)
    return logL,means

def _gm_pi_weights(pi):
    count=GM_SCORE_STATES.sum(1)
    if pi==0.:
        weights=np.full(256,-np.inf);weights[0]=0.;return weights
    if pi==1.:
        weights=np.full(256,-np.inf);weights[-1]=0.;return weights
    return count*np.log(pi)+(8-count)*np.log1p(-pi)

def _gm_predict(N,Tsource,fit,coef,family):
    if fit['constant'] is not None:return np.full(len(N),fit['constant'])
    raw=N@fit['beta']+Tsource@coef
    return expit(raw) if family=='detection' else np.maximum(raw,0.)

def _gm_final_prediction(N,S,Tn,Ts,Y,blocks,betaq,betam):
    shape=(len(Tn),Y.shape[1]);out={key:np.zeros(shape) for key in [
        'observed_prediction','observed_zero_prediction','positive_prediction','positive_zero_prediction',
        'detection_prediction','detection_zero_prediction']};fits=[];zero=np.zeros_like(Ts)
    for g in range(Y.shape[1]):
        keep=Y[:,g]>0
        qfit=_gm_nuisance(N,keep.astype(float),blocks,S@betaq[g],'detection')
        mfit=_gm_nuisance(N[keep],Y[keep,g],blocks[keep],S[keep]@betam[g],'positive')
        q=_gm_predict(Tn,Ts,qfit,betaq[g],'detection');q0=_gm_predict(Tn,zero,qfit,betaq[g],'detection')
        m=_gm_predict(Tn,Ts,mfit,betam[g],'positive');m0=_gm_predict(Tn,zero,mfit,betam[g],'positive')
        for key,value in [('observed_prediction',q*m),('observed_zero_prediction',q0*m0),
            ('positive_prediction',m),('positive_zero_prediction',m0),('detection_prediction',q),('detection_zero_prediction',q0)]:out[key][:,g]=value
        fits.append({'response':g,'q_nuisance':qfit,'m_nuisance':mfit,
            'q_sources':betaq[g].tolist(),'m_sources':betam[g].tolist()})
    return out,fits

from scipy.optimize import minimize,linprog

if True:
    _gm_numerical_original_nuisance=_gm_nuisance

def _gmn_same_objective_fallback(N,y,blocks,offset,old_error):
    n,p=N.shape;w=_gm_weights(blocks)
    # Rank is based on the original design, not a saturated intermediate Hessian.
    U,s,Vt=np.linalg.svd(N,full_matrices=False)
    rank_tolerance=np.finfo(float).eps*max(N.shape)*float(s.max())
    keep=s>rank_tolerance
    if not keep.any():raise FloatingPointError('Fallback nuisance design has no numerical rank')
    Q=U[:,keep];singular=s[keep];right=Vt[keep]
    initial=np.zeros(p)
    initial[0]=logit(np.clip(w@y/w.sum(),1e-10,1-1e-10))-w@offset/w.sum()
    initial_gamma=singular*(right@initial)
    null_initial=initial-right.T@(right@initial)
    # Quasi/complete separation depends on the design/labels, not finite offset.
    # With orthonormal Q a separating unit direction has sum margins >=1,
    # allowing a clear numerical zero check on this bounded linear program.
    signed=(2*y-1)[:,None]*Q
    separation=linprog(-signed.sum(0),A_ub=-signed,b_ub=np.zeros(n),
        bounds=[(-1.,1.)]*Q.shape[1],method='highs',
        options={'primal_feasibility_tolerance':1e-9,'dual_feasibility_tolerance':1e-9})
    if not separation.success:raise FloatingPointError('Fallback separation check failed: '+str(separation.message))
    separation_margin=float(-separation.fun)
    if separation_margin>1e-7:
        raise FloatingPointError('Nuisance logistic separation detected; finite-MLE fallback refused')
    def objective(gamma):
        eta=Q@gamma+offset
        return float(w@(np.logaddexp(0.,eta)-y*eta))
    def gradient(gamma):return Q.T@(w*(expit(Q@gamma+offset)-y))
    def hessian(gamma):
        probability=expit(Q@gamma+offset)
        return Q.T@((w*probability*(1-probability))[:,None]*Q)
    initial_objective=objective(initial_gamma)
    result=minimize(objective,initial_gamma,jac=gradient,hess=hessian,method='trust-exact',
        options={'maxiter':2000,'gtol':1e-10})
    beta=right.T@(result.x/singular)+null_initial
    eta=N@beta+offset;probability=expit(eta)
    raw_gradient=N.T@(w*(probability-y))
    gradient_max=float(np.max(abs(raw_gradient)))
    final_objective=float(w@(np.logaddexp(0.,eta)-y*eta))
    predictor_error=float(np.max(abs(N@beta-Q@result.x)))
    converged=gradient_max<2e-8*max(w.sum(),1.)
    objective_ok=np.isfinite(final_objective) and final_objective<=initial_objective+1e-10*max(1.,abs(initial_objective))
    mapping_ok=predictor_error<=1e-8*(1+float(np.max(abs(eta))))
    if not converged or not objective_ok or not mapping_ok or not np.isfinite(beta).all():
        raise FloatingPointError('Same-objective trust-region nuisance fallback failed: '+str({
            'raw_scaled_gradient':gradient_max/max(w.sum(),1.),'optimizer_message':str(result.message),
            'objective_ok':bool(objective_ok),'mapping_ok':bool(mapping_ok)}))
    H=N.T@((w*probability*(1-probability))[:,None]*N)
    _,rank_audit=_gm_pinv(H)
    return {'beta':beta,'constant':None,'converged':True,'rank':rank_audit['rank'],
        'rank_audit':rank_audit,'iterations':100+int(result.nit),
        'scaled_gradient_max':gradient_max/max(w.sum(),1.),
        'fallback':True,'numerical_fallback':{
            'old_failure_reason':str(old_error),'legacy_iteration_limit':100,
            'method':'trust-exact in original-design SVD orthonormal coordinates',
            'objective':'Identical sum_i w_i[logaddexp(0,N_i beta+offset_i)−y_i(N_i beta+offset_i)]',
            'weights_unchanged':True,'offset_unchanged':True,'penalty':0.,
            'original_success_path_unchanged':True,'design_rank':int(keep.sum()),
            'design_singular_values':s.tolist(),'design_rank_tolerance':rank_tolerance,
            'initial_nullspace_component_preserved':True,
            'separating_direction_max_sum_margin':separation_margin,
            'initial_objective':initial_objective,'final_objective':final_objective,
            'raw_scaled_gradient':gradient_max/max(w.sum(),1.),
            'original_gradient_tolerance':2e-8,'predictor_mapping_max_error':predictor_error,
            'optimizer_success':bool(result.success),'optimizer_message':str(result.message),
            'fallback_iterations':int(result.nit),'converged':True}}

def _gm_nuisance(N,y,blocks,offset,family):
    try:
        return _gm_numerical_original_nuisance(N,y,blocks,offset,family)
    except FloatingPointError as error:
        if family!='detection' or str(error)!='Unpenalized nuisance logistic fit did not converge':
            raise
        return _gmn_same_objective_fallback(N,y,blocks,offset,error)

from scipy.optimize import brentq

def _gms_nuisance_covariance(N,S,y,blocks,fit,source_coef,family):
    p=N.shape[1]
    if not len(y) or fit['constant'] is not None:
        return np.zeros((p,p)),np.zeros(p-1,bool),{'constant_or_empty':True}
    w=_gm_weights(blocks);raw=N@np.asarray(fit['beta'])+S@source_coef
    if family=='detection':
        q=expit(raw);variance=q*(1-q);h=w*variance;scale={}
    else:
        sigma2,scale=_gm_noise_scale(N,S,y);variance=np.full(len(y),sigma2);h=w
    H=N.T@(h[:,None]*N);inverse,rank=_gm_pinv(H)
    meat=N.T@((w*w*variance)[:,None]*N)
    covariance=inverse@meat@inverse;covariance=(covariance+covariance.T)/2
    support=np.ptp(N[:,1:],axis=0)>1e-12
    return covariance,support,{'rank_audit':rank,'variance_scale':scale,
        'model_covariance':'Hnn⁺ Nᵀ diag(w² var) N Hnn⁺; independent working errors, source coefficients fixed.',
        'source_posterior_uncertainty_propagated':False,
        'minimum_covariance_eigenvalue':float(np.linalg.eigvalsh(covariance).min())}

def _gms_calibrate(N,S,y,base,source_coef,nuisance_slopes,family):
    out={**base,'beta':np.r_[0.,nuisance_slopes]}
    if not len(y):return out,{'empty_training_branch':True}
    offset=N[:,1:]@nuisance_slopes+S@source_coef
    if family=='detection':
        target=float(np.mean(y))
        if target in [0.,1.]:
            out['constant']=target;out['beta'][0]=float(logit(np.clip(target,1e-12,1-1e-12)))
        else:
            objective=lambda intercept:float(np.mean(expit(intercept+offset))-target)
            out['beta'][0]=brentq(objective,-float(offset.max())-40.,-float(offset.min())+40.,xtol=1e-12)
            out['constant']=None
        predicted=np.full(len(y),out['constant']) if out['constant'] is not None else expit(out['beta'][0]+offset)
        return out,{'target_unweighted_detection_fraction':target,'training_predicted_mean':float(predicted.mean()),
            'calibration_absolute_error':float(abs(predicted.mean()-target)),'slope_or_source_changed_by_calibration':False}
    out['beta'][0]=float(np.mean(y-offset));out['constant']=None
    raw=out['beta'][0]+offset
    return out,{'target_unweighted_positive_mean':float(y.mean()),'training_raw_predicted_mean':float(raw.mean()),
        'calibration_absolute_error':float(abs(raw.mean()-y.mean())),
        'training_clipped_predicted_mean':float(np.maximum(raw,0.).mean()),
        'training_negative_fraction':float(np.mean(raw<0)),
        'slope_or_source_changed_by_calibration':False}

def _gms_background_predictions(design,Ytrain,base_fits,names):
    Z=np.asarray(design['Z'],float);T=np.asarray(design['T'],float);Y=np.asarray(Ytrain,float)
    source=np.asarray(design['source'],bool);blocks=np.asarray(design['blocks']).astype(str)
    N,S=Z[:,~source],Z[:,source];Tn,Ts=T[:,~source],T[:,source]
    assert N.shape[1]==5 and source.sum()==8
    beta=np.zeros((2,Y.shape[1],4));variance=np.zeros_like(beta);support=np.zeros_like(beta,bool);cov_audits=[]
    for g,old in enumerate(base_fits):
        keep=Y[:,g]>0;ga=[]
        for branch,(short,family,n,s,y,b) in enumerate([
                ('q','detection',N,S,keep.astype(float),blocks),
                ('m','positive',N[keep],S[keep],Y[keep,g],blocks[keep])]):
            fit=old[short+'_nuisance'];src=np.asarray(old[short+'_sources'])
            cov,valid,audit=_gms_nuisance_covariance(n,s,y,b,fit,src,family)
            beta[branch,g]=np.asarray(fit['beta'])[1:];variance[branch,g]=np.maximum(np.diag(cov)[1:],0.)
            support[branch,g]=valid;ga.append(audit)
        cov_audits.append(ga)
    posterior=np.zeros_like(beta);priors=[]
    for branch in range(2):
        mean,_,_,prior=_js_uni_prior(beta[branch].ravel(),variance[branch].ravel(),support[branch].ravel())
        posterior[branch]=mean.reshape(Y.shape[1],4);priors.append(prior)
    shape=(len(T),Y.shape[1]);fields=['observed_prediction','observed_zero_prediction','positive_prediction',
        'positive_zero_prediction','detection_prediction','detection_zero_prediction']
    output={name:{field:np.zeros(shape) for field in fields} for name in names}
    zero=np.zeros_like(Ts)
    for name in names:
        audit_responses=[]
        for g,old in enumerate(base_fits):
            qsource=np.asarray(old['q_sources']);msource=np.asarray(old['m_sources']);keep=Y[:,g]>0
            qfit,qa=_gms_calibrate(N,S,keep.astype(float),old['q_nuisance'],qsource,posterior[0,g],'detection')
            mslopes=beta[1,g] if name=='gm_score_stable_q' else posterior[1,g]
            mfit,ma=_gms_calibrate(N[keep],S[keep],Y[keep,g],old['m_nuisance'],msource,mslopes,'positive')
            q=_gm_predict(Tn,Ts,qfit,qsource,'detection');q0=_gm_predict(Tn,zero,qfit,qsource,'detection')
            m=_gm_predict(Tn,Ts,mfit,msource,'positive');m0=_gm_predict(Tn,zero,mfit,msource,'positive')
            for field,value in [('observed_prediction',q*m),('observed_zero_prediction',q0*m0),
                ('positive_prediction',m),('positive_zero_prediction',m0),('detection_prediction',q),('detection_zero_prediction',q0)]:output[name][field][:,g]=value
            audit_responses.append({'response':g,'q_sources':qsource.tolist(),'m_sources':msource.tolist(),
                'q_nuisance_beta':qfit['beta'].tolist(),'m_nuisance_beta':mfit['beta'].tolist(),
                'q_calibration':qa,'m_calibration':ma,'q_covariance':cov_audits[g][0],'m_covariance':cov_audits[g][1]})
        output[name]['audit']={'model':name,
            'nuisance_EB_q':priors[0],'nuisance_EB_m':None if name=='gm_score_stable_q' else priors[1],
            'prior_pooling':'16 nuisance-slope coefficients (4 per response), separately for each branch; intercept excluded.',
            'conditional_covariance_source_uncertainty_ignored':True,
            'intercept_calibration':'q_marginal_mean_and_m_raw_mean',
            'positive_intercept_target':'Unweighted raw linear mean; clipped mean separately audited.',
            'responses':audit_responses,'test_outcomes_or_truth_used':False}
        for field in fields:assert np.isfinite(output[name][field]).all() and np.all(output[name][field]>=0)
    return output

def _gms_profile_fixed(summary):
    I=np.asarray(summary['information_I']);U=np.asarray(summary['pseudo_score'])
    V=np.asarray(summary['covariance_model']);inverse,rank=_gm_pinv(I)
    beta=inverse@U;cov=inverse@V@inverse;cov=(cov+cov.T)/2
    variance=np.maximum(np.diag(cov),0.)
    ratio=np.divide(3.841458820694124*variance,beta*beta,out=np.full(8,np.inf),where=beta!=0)
    factor=np.maximum(0.,1-ratio)
    return beta*factor,{'minimum_norm_coefficients':beta.tolist(),'model_covariance':cov.tolist(),
        'fixed_factors':factor.tolist(),'information_rank':rank,'constant_c':3.841458820694124,
        'scope':'Same final iter3 local profile summaries; minimum-norm inverse in rank-deficient cases. Coordinatewise fixed shrinkage, not a calibrated test.'}

from pathlib import Path

import copy, hashlib, json

import numpy as np

import pandas as pd

from scipy.special import expit, logit

from scipy.spatial import cKDTree

GM_BROAD_CHANNELS=('clu','fasn','stk39','oligo2')

GM_BROAD_RESPONSES=('Vps35l','Dock10','Gpr37','Sparc')

_GM_BROAD_GEOMETRY_CACHE={}

def _gb_rng(*parts):
    digest=hashlib.sha256('|'.join(map(str,parts)).encode()).digest()
    return np.random.default_rng(np.random.SeedSequence(np.frombuffer(digest,dtype='<u4')))

def _gb_hash(array):
    return hashlib.sha256(np.asarray(array,dtype='<f8').tobytes()).hexdigest()

def _gb_geometry(chip, prepared=None):
    xy=chip['xy']; component=chip['component']; blocks=chip['blocks']; boxes=[]
    for label in np.unique(blocks):
        take=blocks==label; low=xy[take].min(0); high=xy[take].max(0)
        pad=np.maximum(1.-(high-low),0.)/2
        boxes.append({'block':str(label),'component':int(component[take][0]),'low':low-pad,'high':high+pad})
    return {'xy':xy,'component':component,'blocks':blocks,'boxes':boxes,'audit':{'source':'Bundled receiver geometry'}}


def _gb_source_layout(geometry,rep,seed,chip_id):
    """Uniform occupied-block boxes plus two truncated Gaussian source clusters."""
    rng=_gb_rng(seed,'relocated-source-layout-v1',int(rep),chip_id)
    boxes=geometry['boxes'];records=[];rejected=0
    for channel in GM_BROAD_CHANNELS:
        total=int(rng.integers(25,41));nrandom=total//2;ncluster=total-nrandom
        for j in range(nrandom):
            box=boxes[int(rng.integers(len(boxes)))];xy=rng.uniform(box['low'],box['high'])
            records.append({'channel':channel,'kind':'uniform','cluster':-1,'component':box['component'],
                'anchor_block':box['block'],'x_um':float(xy[0]),'y_um':float(xy[1])})
        choices=rng.choice(len(boxes),size=min(2,len(boxes)),replace=False)
        centers=[(boxes[int(k)],rng.uniform(boxes[int(k)]['low'],boxes[int(k)]['high'])) for k in choices]
        for j in range(ncluster):
            cluster=j%len(centers);box,center=centers[cluster]
            for attempt in range(100000):
                xy=center+rng.normal(0.,30.,2)
                if np.all(xy>=box['low']) and np.all(xy<=box['high']):break
                rejected+=1
            else:raise RuntimeError('Synthetic source cluster rejection limit reached')
            records.append({'channel':channel,'kind':'cluster','cluster':int(cluster),'component':box['component'],
                'anchor_block':box['block'],'x_um':float(xy[0]),'y_um':float(xy[1])})
    return pd.DataFrame(records),rejected

def _gb_annuli(receiver_xy,receiver_component,sources):
    near=np.zeros((len(receiver_xy),4),np.int64);far=np.zeros_like(near)
    nearest=np.ones(len(receiver_xy),float)
    for component in np.unique(receiver_component):
        take=receiver_component==component;R=receiver_xy[take]
        local=sources.loc[sources.component.eq(component)]
        if len(local):nearest[take]=np.minimum(cKDTree(local[['x_um','y_um']].to_numpy()).query(R,k=1)[0]/100.,1.)
        for j,channel in enumerate(GM_BROAD_CHANNELS):
            S=local.loc[local.channel.eq(channel),['x_um','y_um']].to_numpy()
            if not len(S):continue
            tree=cKDTree(S)
            c20=tree.query_ball_point(R,r=20.,return_length=True)
            c50=tree.query_ball_point(R,r=50.,return_length=True)
            c100=tree.query_ball_point(R,r=100.,return_length=True)
            near[take,j]=c50-c20;far[take,j]=c100-c50
    return near,far,nearest

def broad_layout(chip,layout,rep,protocol,prepared_dir):
    """Return (new_chip,audit); all layouts preserve actual receiver/density."""
    if layout not in ('original','relocated'):raise ValueError('Unknown layout')
    out=copy.deepcopy(chip)
    if layout=='original':
        audit={'layout':'original','replicate':int(rep),'chip_id':chip['chip_id'],
            'counts_near_sha256':_gb_hash(chip['counts_near']),'counts_far_sha256':_gb_hash(chip['counts_far']),
            'density_unchanged':True,'synthetic_source_positions':False}
    else:
        geometry=_gb_geometry(chip,prepared_dir)
        sources,rejected=_gb_source_layout(geometry,rep,protocol['seed'],chip['chip_id'])
        near,far,nearest=_gb_annuli(geometry['xy'],geometry['component'],sources)
        out['counts_near']=near;out['counts_far']=far
        for j,channel in enumerate(GM_BROAD_CHANNELS):
            for ring,counts in [('20_50um',near),('50_100um',far)]:
                out['B3'][f'target_annulus::{channel}::{ring}']=np.log1p(counts[:,j])
        for table in ['B2','B3']:
            for ring,counts in [('20_50um',near),('50_100um',far)]:
                out[table][f'pooled_annulus::{ring}']=np.log1p(counts.sum(1))
                out[table][f'multi_annulus::{ring}']=0.
            out[table]['nearest_barcode_distance_capped_100um']=nearest
            for col in ['density::50um','density::100um']:
                assert np.array_equal(out[table][col].to_numpy(),chip[table][col].to_numpy())
        source_csv=sources.to_csv(index=False,float_format='%.17g')
        audit={'layout':'relocated','replicate':int(rep),'chip_id':chip['chip_id'],'seed_namespace':str(protocol['seed']),
            'seed_parts':[str(protocol['seed']),'relocated-source-layout-v1',int(rep),chip['chip_id']],
            'source_counts':{k:int(v) for k,v in sources.channel.value_counts().sort_index().items()},
            'source_positions_sha256':hashlib.sha256(source_csv.encode()).hexdigest(),
            'source_positions':sources.to_dict('records'),'cluster_rejected_proposals':rejected,
            'counts_near_sha256':_gb_hash(near),'counts_far_sha256':_gb_hash(far),
            'receiver_geometry':geometry['audit'],'density_unchanged':True,'receiver_blocks_unchanged':True,
            'source_receiver_separation':'New continuous synthetic source points; no receiver cell relabelled or removed.',
            'sampling':'Per channel UniformInteger[25,40]; floor(N/2) points uniform over uniformly sampled occupied evaluation-block bounding boxes; remainder in2centres, isotropic sigma30um Gaussian truncated to centre block box.',
            'tissue_support':'Occupied receiver-block bounding boxes approximate the footprint, not a measured tissue mask.',
            'annuli':'(20,50] and (50,100] micrometres; component-restricted KD-tree; pooled=sum4channels.',
            'other_features':'Actual coordinates/density unchanged; nearest recomputed; multi-barcode=0 because synthetic sources have one channel each.',
            'biological_scope':'Synthetic covariate relocation on the same receiver clouds; not new animals or wet-lab data.'}
    out['broad_layout_audit']=audit
    out['provenance']={**out.get('provenance',{}),'broad_synthetic_layout':{k:v for k,v in audit.items() if k!='source_positions'}}
    return out,audit

def _gb_fields(chip):
    near=np.asarray(chip['counts_near'],float);S=near/(1.+near)
    pool=np.expm1(chip['B2']['pooled_annulus::20_50um'].to_numpy(float))
    density=np.expm1(chip['B2']['density::50um'].to_numpy(float))
    if S.shape[1]!=4 or np.any(near<0):raise ValueError('Four nonnegative source channels required')
    base=2.+.2*pool/(1.+pool)+.1*density/(1.+density)
    return S,np.broadcast_to(base[:,None],S.shape).copy()

def _gb_outcome(chip,mu,q,zero_mu,zero_q,rng,draw_audit):
    if np.any(mu<=.25) or np.any(zero_mu<=.25):raise ValueError('Positive continuous outcome support violated')
    U=mu+rng.uniform(-.25,.25,mu.shape);Y=U*(rng.random(mu.shape)<q)
    observed=q*mu;zero=zero_q*zero_mu;shift=mu-zero_mu
    constant=float(q.flat[0]) if np.all(q==q.flat[0]) else None
    data={'Y':Y,'latent_mean':mu,'observed_mean':observed,'positive_mean':mu.copy(),
        'latent_shift':shift,'prob_positive':q,'constant_capture':constant,
        'Y_sha256':_gb_hash(Y),'draw_audit':draw_audit}
    truth={'observed_zero':zero,'observed_effect':observed-zero,
        'variance':q*(.25**2/3)+q*(1-q)*mu**2,'latent_effect':shift,'positive_equals_latent':True}
    return data,truth

def gm_factorial_generate(chip,p,aq,am,rep,protocol,layout):
    """16 q-by-m-by-p conditions per layout; matched4source-response mapping."""
    p=float(p);aq=float(aq);am=float(am)
    if not 0<p<1:raise ValueError('p must be interior')
    if layout not in ('original','relocated'):raise ValueError('Unknown layout')
    metadata=chip.get('broad_layout_audit')
    if metadata is not None and metadata['layout']!=layout:raise ValueError('Layout metadata differs from outcome request')
    S,base=_gb_fields(chip);q=expit(logit(p)+aq*S);mu=base+am*S
    parts=[protocol['seed'],'broad-factorial-v1',layout,p,aq,am,int(rep),chip['chip_id']]
    audit={'generator':'factorial','seed_parts':list(map(str,parts)),'layout':layout,'p':p,'aq':aq,'am':am,
        'replicate':int(rep),'channel_response_mapping':dict(zip(GM_BROAD_CHANNELS,GM_BROAD_RESPONSES)),
        'mapping_scope':'Invented one-to-one synthetic source-response matches inherited from js_generate.',
        'source_zero':'Only8identity-specific near/far covariates set0; pooled/density background fixed.',
        'conditional_sampling':'U=m+Uniform(-.25,.25), Y=Bernoulli(q)*U; not UMI counts.'}
    return _gb_outcome(chip,mu,q,base,np.full_like(base,p),_gb_rng(*parts),audit)

def gm_new_stress_generate(chip,rep,protocol,mechanism):
    S,base=_gb_fields(chip);p=.08
    layout=chip.get('broad_layout_audit',{}).get('layout','original')
    parts=[protocol['seed'],'broad-new-stress-v1',mechanism,layout,int(rep),chip['chip_id']]
    rng=_gb_rng(*parts);extra={}
    if mechanism=='block_correlated':
        labels,inverse=np.unique(np.asarray(chip['blocks']).astype(str),return_inverse=True)
        # Uniform chosen before confirmation to guarantee positive support for all seeds.
        offsets=rng.uniform(-.25,.25,(len(labels),4));u=offsets[inverse]
        mu=base+S+u;zero_mu=base+u;q=np.full_like(mu,p)
        extra={'block_offset_distribution':'Independent block-by-response Uniform(-.25,.25), shared within each block',
            'block_offsets_sha256':_gb_hash(offsets),'offset_min':float(offsets.min()),'offset_max':float(offsets.max()),
            'source_zero_preserves_same_block_offset':True,
            'truth_conditioning':'Mean/variance condition on realized block effect; observations are correlated marginally over block draws.'}
    elif mechanism=='nonlinear_saturation':
        mu=base+.8*np.sqrt(S);zero_mu=base;q=expit(logit(p)+2*S*S)
        extra={'q_formula':'expit(logit(.08)+2*S^2)','m_formula':'base+.8*sqrt(S)'}
    else:raise ValueError('Unknown new stress')
    audit={'generator':mechanism,'seed_parts':list(map(str,parts)),'layout':layout,'replicate':int(rep),**extra}
    return _gb_outcome(chip,mu,q,zero_mu,np.full_like(base,p),rng,audit)

import numpy as np

from scipy.optimize import brentq, minimize

CS_FACTORIAL_NAMES=['cs_B'+str(b)+str(v)+str(s) for b in range(2) for v in range(2) for s in range(2)]

def _cs_screen(history,Y,blocks,enabled):
    """Final pooled-branch EB evidence guard; no changes to the three updates."""
    branch_masks={'q':np.ones(len(Y),bool),'m':np.any(Y>0,axis=1)}
    decisions={}
    for short,mask in branch_masks.items():
        prior=history[-1][short+'_prior']
        logbf=max(float(row['log_marginal_BF_vs_null']) for row in prior['grid_candidates'])
        B=int(len(np.unique(blocks[mask])));threshold=float(np.log(max(B,2)))
        passed=bool(B>0 and logbf>threshold)
        decisions[short]={'enabled':bool(enabled),'log_marginal_BF_vs_all_null':logbf,
            'threshold_log_max_B_2':threshold,'occupied_training_blocks':B,
            'block_definition':'All training receiver blocks' if short=='q' else 'Union of blocks with a positive observation in at least one of the four responses',
            'evidence_pass':passed,'source_branch_zeroed':bool(enabled and not passed),
            'screen_stage':'After the third local update; pooled response branch, before final nuisance refit',
            'interpretation':'BIC-like guard for the two fitted hyperparameters pi and tau2; not an FWER/FDR test.'}
    return decisions

def _cs_screened_refits(design,Y,parent,screen):
    base=parent['audit']['final_nuisance_refits']
    if not any(s['source_branch_zeroed'] for s in screen.values()):return base
    Z=np.asarray(design['Z'],float);T=np.asarray(design['T'],float)
    source=np.asarray(design['source'],bool);blocks=np.asarray(design['blocks']).astype(str)
    q=np.asarray([fit['q_sources'] for fit in base]);m=np.asarray([fit['m_sources'] for fit in base])
    if screen['q']['source_branch_zeroed']:q=np.zeros_like(q)
    if screen['m']['source_branch_zeroed']:m=np.zeros_like(m)
    _,refits=_gm_final_prediction(Z[:,~source],Z[:,source],T[:,~source],T[:,source],Y,blocks,q,m)
    return refits

import numpy as np

import mpmath as mp

try:
    import gmpy2
except ImportError:
    gmpy2=None

from scipy.optimize import linprog,minimize,brentq

from scipy import sparse

CS_SUPPORT_ROUNDOFF_BUDGET=1e-6

if True:
    _cs_support_original=_gm_support_components

def _cs_native_rank_update(group,tau2):
 A,u,V=group['A'],group['u'],group['V'];r=len(u);means=np.zeros((256,8));logL=np.zeros(256)
 with mp.workdps(90):
  Vm=mp.matrix([[mp.mpf(float(x)) for x in row] for row in V]);base=mp.inverse(Vm);base_log=mp.log(mp.det(Vm));bits=mp.mp.prec
  def convert(x):
   sign,mantissa,exponent,_=x._mpf_;v=gmpy2.mul_2exp(gmpy2.mpfr(mantissa),exponent)
   return -v if sign else v
  with gmpy2.context(precision=bits):
   invs={0:[[convert(base[i,j]) for j in range(r)] for i in range(r)]};ld={0:convert(base_log)}
   cols=[[gmpy2.mpfr(float(A[i,k])) for i in range(r)] for k in range(8)];uv=[gmpy2.mpfr(float(x)) for x in u];tau=gmpy2.mpfr(float(tau2));constant=r*gmpy2.log(2*gmpy2.const_pi())
   for mask in range(256):
    if mask:
     parent=mask&(mask-1);k=(mask^parent).bit_length()-1;old=invs[parent];a=cols[k]
     left=[sum(old[i][j]*a[j] for j in range(r)) for i in range(r)];right=[sum(a[j]*old[j][i] for j in range(r)) for i in range(r)]
     denom=1+tau*sum(a[i]*left[i] for i in range(r))
     if denom<=0:raise ArithmeticError('Nonpositive determinant update')
     factor=tau/denom
     invs[mask]=[[old[i][j]-factor*left[i]*right[j] for j in range(r)] for i in range(r)]
     ld[mask]=ld[parent]+gmpy2.log(denom)
    inv=invs[mask];solved=[sum(inv[i][j]*uv[j] for j in range(r)) for i in range(r)]
    logL[mask]=float(-gmpy2.mpfr('.5')*(constant+ld[mask]+sum(uv[i]*solved[i] for i in range(r))))
    for k in range(8):
     if mask>>k&1:means[mask,k]=float(tau*sum(cols[k][i]*solved[i] for i in range(r)))
 return logL,means

def _cs_support_high_precision(group,tau2):
    A=np.asarray(group['A'],float);u=np.asarray(group['u'],float);V=np.asarray(group['V'],float)
    assert np.isfinite(tau2) and tau2>=0
    if gmpy2 is not None:
        result=_cs_native_rank_update({'A':A,'u':u,'V':V},tau2)
        if not np.isfinite(result[0]).all() or not np.isfinite(result[1]).all():
            raise FloatingPointError('Native high-precision support likelihood is not finite')
        return result
    r=len(u)
    logL=np.zeros(len(GM_SCORE_STATES));means=np.zeros((len(GM_SCORE_STATES),A.shape[1]))
    with mp.workdps(90):
        # mp.mpf(float) preserves the supplied float64 number exactly. Rebuild
        # A_S*A_S.T at high precision rather than converting a rounded Gram.
        Am=mp.matrix([[mp.mpf(float(x)) for x in row] for row in A])
        Vm=mp.matrix([[mp.mpf(float(x)) for x in row] for row in V])
        um=mp.matrix([mp.mpf(float(x)) for x in u]);tm=mp.mpf(float(tau2))
        constant=r*mp.log(2*mp.pi)
        inverses={0:mp.inverse(Vm)};logdets={0:mp.log(mp.det(Vm))}
        columns=[Am[:,k] for k in range(A.shape[1])]
        for index,state in enumerate(GM_SCORE_STATES):
            active=np.flatnonzero(state)
            if index:
                parent=index&(index-1);k=(index^parent).bit_length()-1
                old=inverses[parent];column=columns[k]
                left=old*column;right=column.T*old
                denominator=1+tm*(column.T*left)[0]
                if denominator<=0:raise FloatingPointError('Support determinant update is not positive')
                inverses[index]=old-(tm/denominator)*(left*right)
                logdets[index]=logdets[parent]+mp.log(denominator)
            solved=inverses[index]*um
            logL[index]=float(-mp.mpf('.5')*(constant+logdets[index]+(um.T*solved)[0]))
            for k in active:means[index,int(k)]=float(tm*(columns[int(k)].T*solved)[0])
    if not np.isfinite(logL).all() or not np.isfinite(means).all():
        raise FloatingPointError('High-precision same-covariance support likelihood is not finite')
    return logL,means

def _cs_support_numerical(group,tau2):
    if group is None:return _cs_support_original(group,tau2)
    A=np.asarray(group['A'],float);V=np.asarray(group['V'],float)
    if not np.isfinite(A).all() or not np.isfinite(V).all() or not np.isfinite(group['u']).all():
        raise FloatingPointError('Nonfinite Gaussian support-likelihood inputs cannot be repaired numerically')
    minimum=float(np.linalg.eigvalsh((V+V.T)/2).min())
    norm_V=float(np.linalg.norm(V,2));norm_A=float(np.linalg.norm(A,2))
    with np.errstate(over='ignore',divide='ignore',invalid='ignore'):
        bound=float(np.finfo(float).eps*(norm_V+tau2*norm_A*norm_A)/minimum) if minimum>0 else float('inf')
    trigger=None
    if bound>CS_SUPPORT_ROUNDOFF_BUDGET or not np.isfinite(bound):trigger='roundoff_bound_exceeds_budget'
    elif minimum<=0:trigger='nonpositive_working_covariance'
    else:
        gram=np.einsum('ik,sk,jk->sij',A,GM_SCORE_STATES,A)
        C=V[None]+tau2*gram
        if not np.isfinite(C).all():trigger='nonfinite_formed_covariance'
        else:
            try:np.linalg.cholesky(C)
            except np.linalg.LinAlgError:trigger='formed_covariance_not_positive_definite'
        if trigger is None:
            try:result=_cs_support_original(group,tau2)
            except np.linalg.LinAlgError as error:
                if str(error)!='Singular matrix':raise
                trigger='original_solve_singular'
            else:
                if np.isfinite(result[0]).all() and np.isfinite(result[1]).all():return result
                trigger='original_output_nonfinite'
    result=_cs_support_high_precision(group,tau2)
    audit={'trigger':trigger,'tau2':float(tau2),
            'method':'90-decimal covariance inverse and determinant rank-one updates on the full support lattice',
            'decimal_precision':90,'engine':'native MPFR rank updates' if gmpy2 is not None else 'mpmath rank updates',
            'covariance':'Exactly V + tau2*A_support*A_support.T',
            'added_jitter':0.,'additional_rank_truncation':False,
            'prior_or_grid_changed':False,'accepted_low_bound_original_path_unchanged':True,
            'roundoff_bound':bound if np.isfinite(bound) else 'inf',
            'roundoff_budget':CS_SUPPORT_ROUNDOFF_BUDGET,
            'roundoff_formula':'eps*(spectral_norm(V)+tau2*spectral_norm(A)^2)/lambda_min(sym(V))',
            'roundoff_interpretation':'Conservative arithmetic sensitivity routing proxy, not a proved bound on output error.',
            'V_minimum_eigenvalue':minimum,
            'V_maximum_eigenvalue':float(np.linalg.eigvalsh(group['V']).max()),
            'V_spectral_norm':norm_V,'A_spectral_norm':norm_A,
            'outputs_finite':True}
    group.setdefault('support_component_numerical_fallbacks',[]).append(audit)
    return result

_gm_support_components=_cs_support_numerical

if True:
    _cs_previous_nuisance=_gm_nuisance

def _cs_large_offset_nuisance(N,y,blocks,offset,old_error):
    """Same unpenalized objective after both previous numerical solvers failed."""
    n,p=N.shape;w=_gm_weights(blocks);scale=max(float(np.max(abs(offset))),1.)
    U,s,Vt=np.linalg.svd(N,full_matrices=False)
    cutoff=np.finfo(float).eps*max(N.shape)*float(s.max());keep=s>cutoff
    if not keep.any():raise FloatingPointError('LP-initialized nuisance design has no numerical rank')
    Q=U[:,keep];singular=s[keep];right=Vt[keep]
    old_initial=np.zeros(p)
    old_initial[0]=logit(np.clip(w@y/w.sum(),1e-10,1-1e-10))-w@offset/w.sum()
    null_initial=old_initial-right.T@(right@old_initial)
    offset_fit=offset+N@null_initial;dimension=Q.shape[1]
    # The scaled hinge LP only supplies a starting point. It does not replace
    # the original logistic objective, add a penalty, or constrain its solution.
    sign=2*y-1
    constraints=sparse.hstack([-sparse.csr_matrix(sign[:,None]*Q),-sparse.eye(n)],format='csr')
    lp=linprog(np.r_[np.zeros(dimension),w],A_ub=constraints,b_ub=sign*offset_fit/scale,
        bounds=[(None,None)]*dimension+[(0,None)]*n,method='highs')
    if not lp.success:raise FloatingPointError('Same-objective nuisance hinge initializer failed: '+lp.message)
    initial=scale*lp.x[:dimension]
    def objective(beta):
        raw=Q@beta+offset_fit
        return float(w@(np.logaddexp(0.,raw)-y*raw))
    def gradient(beta):return Q.T@(w*(expit(Q@beta+offset_fit)-y))
    def hessian(beta):
        q=expit(Q@beta+offset_fit)
        return Q.T@((w*q*(1-q))[:,None]*Q)
    initial_objective=objective(initial);attempts=[];accepted=None
    for method in ['BFGS','trust-exact']:
        options={'maxiter':2000,'gtol':1e-10}
        extra={'hess':hessian} if method=='trust-exact' else {}
        if method=='trust-exact':options.update(initial_trust_radius=1.,max_trust_radius=max(1000.,10*scale))
        result=minimize(objective,initial,jac=gradient,method=method,options=options,**extra)
        beta=right.T@(result.x/singular)+null_initial
        raw=N@beta+offset;probability=expit(raw)
        scaled=float(np.max(abs(N.T@(w*(probability-y))))/max(w.sum(),1.))
        value=float(w@(np.logaddexp(0.,raw)-y*raw))
        mapping_error=float(np.max(abs(N@beta-(Q@result.x+N@null_initial))))
        okay=bool(np.isfinite(beta).all() and np.isfinite(value) and scaled<2e-8 and value<=initial_objective+1e-10*max(1.,abs(initial_objective))
            and mapping_error<=1e-8*(1+float(np.max(abs(raw)))))
        attempts.append({'method':method,'iterations':int(result.nit),'success':bool(result.success),
            'message':str(result.message),'raw_scaled_gradient':scaled,'objective':value,'predictor_mapping_max_error':mapping_error,'accepted':okay})
        if okay:accepted=(result,beta);break
    if accepted is None:raise FloatingPointError('Same-objective LP-initialized nuisance solver failed: '+repr(attempts))
    result,beta=accepted;probability=expit(N@beta+offset)
    H=N.T@((w*probability*(1-probability))[:,None]*N);_,rank=_gm_pinv(H)
    return {'beta':beta,'constant':None,'converged':True,'rank':rank['rank'],'rank_audit':rank,
        'iterations':int(result.nit),'scaled_gradient_max':attempts[-1]['raw_scaled_gradient'],'fallback':True,
        'numerical_fallback':{'old_failure_reason':str(old_error),'method':'Scaled weighted hinge LP initialization, then original logistic objective',
            'penalty':0.,'weights_and_offset_unchanged':True,'previous_separation_check_passed':True,
            'original_design_rank':dimension,'design_rank_tolerance':cutoff,'original_initial_nullspace_component_preserved':True,
            'LP_scale':scale,'LP_success':bool(lp.success),'initial_objective':initial_objective,
            'final_objective':attempts[-1]['objective'],'original_gradient_tolerance':2e-8,'attempts':attempts}}

def _gm_nuisance(N,y,blocks,offset,family):
    try:return _cs_previous_nuisance(N,y,blocks,offset,family)
    except FloatingPointError as error:
        if family!='detection' or not str(error).startswith('Same-objective trust-region nuisance fallback failed:'):raise
        return _cs_large_offset_nuisance(N,y,blocks,offset,error)

from scipy.optimize import minimize

from scipy.special import logsumexp

CS_STANDARD_NAMES=['logistic_hurdle','ridgebase_multiscale_normal_EB']

def cs_multiscale_normal_eb(beta,variance,supported):
    """Convex marginal-likelihood mixture; no significance/FDR output."""
    beta=np.asarray(beta,float);variance=np.asarray(variance,float);supported=np.asarray(supported,bool)
    means=np.zeros_like(beta);posterior_variance=np.zeros_like(beta);idx=np.flatnonzero(supported)
    if not len(idx):return means,posterior_variance,{'supported':0,'all_null':True,'weights':[1.],'tau2_grid':[0.]}
    b=beta[idx];v=np.maximum(variance[idx],1e-12)
    scale=max(float(np.max(b*b)),float(np.median(v)),1e-12)
    tau=np.r_[0.,scale*np.geomspace(1e-3,4.,17)]
    logL=-.5*(np.log(2*np.pi*(v[:,None]+tau[None,:]))+b[:,None]**2/(v[:,None]+tau[None,:]))
    offset=logL.max(1);L=np.exp(logL-offset[:,None]);K=len(tau)
    # The matched two-component optimum is a feasible mixture and a declared start.
    _,_,_,spike=_js_uni_prior(beta,variance,supported)
    initial=np.zeros(K);j=int(np.argmin(abs(tau-float(spike['tau2']))))
    if j==0:initial[0]=1.
    else:initial[0]=1-float(spike['pi_slab']);initial[j]=float(spike['pi_slab'])
    def objective(pi):
        den=np.maximum(L@pi,1e-300)
        return -float(np.log(den).sum()),-np.sum(L/den[:,None],axis=0)
    options=[];numerical=[]
    for start in [np.full(K,1/K),initial]:
        opt=minimize(objective,start,jac=True,method='SLSQP',bounds=[(0.,1.)]*K,
            constraints={'type':'eq','fun':lambda x:x.sum()-1.,'jac':lambda x:np.ones(K)},
            options={'ftol':1e-11,'maxiter':500})
        pi=np.maximum(opt.x,0.);pi/=pi.sum();options.append(pi)
        numerical.append({'success_flag':bool(opt.success),'iterations':int(opt.nit),'message':str(opt.message)})
    options += [initial]+[np.eye(K)[j] for j in range(K)]
    pi=min(options,key=lambda x:objective(x)[0]);gap=float(max(0.,(-objective(pi)[1]).max()+objective(pi)[1]@pi))
    if gap>1e-5:
        # EM is a deterministic numerical repair, not a new statistical rule.
        trial=np.maximum(pi,1e-12);trial/=trial.sum()
        for iteration in range(20000):
            den=np.maximum(L@trial,1e-300);g=np.sum(L/den[:,None],axis=0)
            trial*=g/len(b);trial/=trial.sum()
            gap_trial=float(max(0.,g.max()-g@trial))
            if gap_trial<1e-7:break
        if objective(trial)[0]<objective(pi)[0]:pi=trial
        gap=float(max(0.,(-objective(pi)[1]).max()+objective(pi)[1]@pi))
        numerical.append({'repair':'multiplicative EM','iterations':iteration+1})
    if gap>1e-4:raise FloatingPointError('Multiscale-normal mixture KKT gap '+str(gap))
    posterior=pi[None,:]*L;posterior/=posterior.sum(1)[:,None]
    factor=tau[None,:]/(v[:,None]+tau[None,:]);component_mean=factor*b[:,None]
    mean=np.sum(posterior*component_mean,axis=1)
    second=np.sum(posterior*(factor*v[:,None]+component_mean**2),axis=1)
    means[idx]=mean;posterior_variance[idx]=np.maximum(second-mean*mean,0.)
    loglik=float(-objective(pi)[0]+offset.sum());single=float(-objective(initial)[0]+offset.sum())
    assert loglik>=single-1e-8
    return means,posterior_variance,{'supported':len(idx),'pooled_units':len(beta),'weights':pi.tolist(),
        'tau2_grid':tau.tolist(),'marginal_loglikelihood':loglik,'matched_spike_loglikelihood':single,
        'KKT_simplex_gap':gap,'numerical_optimization':numerical,'variance_floor':1e-12,
        'prior':'Point mass0 plus17zero-centered Gaussian slabs; freely estimated mixture proportions, no null-biased Dirichlet penalty.',
        'grid':'Exactly the matched Independent-spike training-scaled tau2 grid, plus0.',
        'input_scope':'Working normal approximation to penalized Ridge/logistic source coefficients with estimated CR0 diagonal; cross-source dependence ignored.',
        'software_label':'Self-implemented ASH-style normal-mixture EB; official ashr was not invoked. Not a full/default ASH fit; no FDR claim.',
        'sources':['https://doi.org/10.1093/biostatistics/kxw041','https://search.r-project.org/CRAN/refmans/ashr/html/ash.html']}

def fit_cs_standard_eb(design,Ytrain,pack=None):
    """Two standard comparators, all hyperparameters selected on training only."""
    Z=np.asarray(design['Z'],float);T=np.asarray(design['T'],float);Y=np.asarray(Ytrain,float)
    blocks=np.asarray(design['blocks']).astype(str);source=np.asarray(design['source'],bool)
    packs,b,cov,support=_js_base_pack(Z,Y,blocks,source) if pack is None else pack
    G=Y.shape[1];K=int(source.sum());assert K==8
    coefficients=[];priors=[]
    for branch in range(2):
        coef,_,prior=cs_multiscale_normal_eb(b[:,branch],cov[:,branch,branch],support[:,branch])
        coefficients.append(coef);priors.append(prior)
    out={name:{field:np.zeros((len(T),G)) for field in BA_FIELDS} for name in CS_STANDARD_NAMES};audits=[]
    for g,item in enumerate(packs):
        qb,mb,keep=item['q'],item['m'],item['keep'];ix=slice(g*K,(g+1)*K)
        qfit=_js_refit(qb,Z,keep.astype(float),blocks,source,coefficients[0][ix])
        mfit=_js_refit(mb,Z[keep],Y[keep,g],blocks[keep],source,coefficients[1][ix])
        for name,(q,m) in [('logistic_hurdle',(qb,mb)),('ridgebase_multiscale_normal_EB',(qfit,mfit))]:
            pred=_js_prediction(q,m,T,source)
            for field in BA_FIELDS:out[name][field][:,g]=pred[field]
        audits.append({'response':g,'q_alpha':item['q_alpha'],'m_alpha':item['m_alpha'],
            'q_base_coefficients':qb['beta'].tolist(),'m_base_coefficients':mb['beta'].tolist(),
            'q_EB_coefficients':qfit['beta'].tolist(),'m_EB_coefficients':mfit['beta'].tolist(),
            'q_constant':qb['constant'],'m_constant':mb['constant'],
            'q_nuisance_refit':qfit.get('refit_audit',{}),'m_nuisance_refit':mfit.get('refit_audit',{})})
    for name in CS_STANDARD_NAMES:
        out[name]['audit']={'model':name,'base_fit':'Per-response logistic detection and positive linear Ridge; seven-alpha whole-training-block CV.',
            'source_summary':'Same basepack and CR0 diagonal as Independent spike','responses':audits,
            'prior':priors if name=='ridgebase_multiscale_normal_EB' else None,
            'test_outcomes_or_truth_used':False,'target':'Conditional hurdle mean and source-zero contrast; not a causal identification claim.'}
        for field in BA_FIELDS:assert np.isfinite(out[name][field]).all() and np.all(out[name][field]>=0)
    return out

import hashlib,json,time,copy

from pathlib import Path

import numpy as np

import pandas as pd

from scipy.special import logsumexp

from scipy.optimize import minimize_scalar

def cx_rng(*parts):
    h=hashlib.sha256('|'.join(map(str,parts)).encode()).digest()
    return np.random.default_rng(np.random.SeedSequence(np.frombuffer(h,dtype='<u4')))

def cx_diag_components(group,tau):
    # L(beta) ∝ exp(h' beta - beta'H beta/2), H=A'V^-1 A.
    # Remove ONLY parameter cross-terms; retain h, diag(H), support prior,
    # shared scale grid and prior estimation budget. This is not ASH software.
    if group is None:return np.zeros(256),np.zeros((256,8))
    A,u,V=group['A'],group['u'],group['V']
    solved=np.linalg.solve(V,np.column_stack([A,u]))
    d=np.einsum('ij,ij->j',A,solved[:,:8]);h=A.T@solved[:,8]
    assert d.min()>-1e-10*max(d.max(),1.)
    d=np.maximum(d,0.)
    scale=1+tau*d
    bf=-.5*np.log1p(tau*d)+.5*tau*h*h/scale
    return GM_SCORE_STATES@bf,GM_SCORE_STATES*(tau*h/scale)[None,:]

def cx_prior(summaries,covariance,mode):
    assert mode in ['full','diagonal']
    groups=[_gm_likelihood_group(s,covariance) for s in summaries]
    scale=1e-8
    for s in summaries:
        d=np.diag(s['information_I']);valid=d>max(float(d.max()),1e-30)*1e-10
        if valid.any():scale=max(scale,float(np.max((s['pseudo_score'][valid]/d[valid])**2)),float(np.max(np.diag(s['covariance_'+covariance])[valid]/d[valid]**2)))
    taus=scale*np.geomspace(1e-3,10.,9);grid=np.linspace(0,1,41)
    logweights=np.array([_gm_pi_weights(p) for p in grid]);best=(-np.inf,None);candidates=[]
    for tau in taus:
        if mode=='full':
            components=[_gm_support_components(g,float(tau)) for g in groups]
            ll=np.array([c[0]-c[0][0] for c in components])
        else:
            components=[cx_diag_components(g,float(tau)) for g in groups]
            ll=np.array([c[0] for c in components])
        values=logsumexp(ll[None,:,:]+logweights[:,None,:],axis=2).sum(1)
        index=int(np.argmax(values));pi=float(grid[index]);value=float(values[index])
        objective=lambda p:-float(logsumexp(ll+_gm_pi_weights(float(p)),axis=1).sum())
        opt=minimize_scalar(objective,bounds=(grid[max(index-1,0)],grid[min(index+1,40)]),method='bounded',options={'xatol':1e-8})
        if opt.success and -opt.fun>value+1e-10:pi,value=float(opt.x),float(-opt.fun)
        candidates.append({'tau2':float(tau),'pi':pi,'log_marginal_BF_vs_null':value})
        if value>best[0]+1e-10:best=(value,(pi,float(tau),ll,components))
    pi,tau,ll,components=best[1];means=[];pip=[]
    for g,(_,component),l in zip(groups,components,ll):
        responsibility=np.exp(l+_gm_pi_weights(pi)-logsumexp(l+_gm_pi_weights(pi)))
        means.append(responsibility@component if pi>0 else np.zeros(8))
        pip.append(responsibility@GM_SCORE_STATES if g is not None else np.zeros(8))
    return np.asarray(means),{'pi_slab':pi,'tau2':tau if pi else 0.,'scale_grid':scale,'grid_candidates':candidates,
        'posterior_inclusion':np.asarray(pip).tolist(),'posterior_source_means':np.asarray(means).tolist(),
        'mode':mode,'same_prior_family_and_grid':True,'max_logBF':float(best[0])}

def cx_design(chip,counts=None):
    frame=_cd_bounded(chip);Z=np.column_stack([np.ones(len(frame)),frame.to_numpy()])
    if counts is not None:
        counts=np.asarray(counts,float);assert counts.shape==(len(Z),8) and np.all(counts>=0)
        Z[:,5:]=counts/(1+counts)
    return {'Z':Z,'T':Z.copy(),'source':np.r_[np.zeros(5,bool),np.ones(8,bool)],
            'blocks':np.asarray(chip['blocks']).astype(str)}

def cx_hurdle(design,Y,mode):
    Z=design['Z'];N,S=Z[:,:5],Z[:,5:];blocks=design['blocks'];labels=np.unique(blocks)
    q=np.zeros((4,8));m=np.zeros_like(q);history=[]
    for iteration in range(3):
        qs=[];ms=[]
        for g in range(4):
            keep=Y[:,g]>0
            qs.append(_gm_profile(N,S,keep.astype(float),blocks,q[g],'detection',labels))
            ms.append(_gm_profile(N[keep],S[keep],Y[keep,g],blocks[keep],m[g],'positive',labels))
        q,qp=cx_prior(qs,'model',mode);m,mp=cx_prior(ms,'model',mode)
        history.append({'iteration':iteration+1,'q_prior':qp,'m_prior':mp})
    _,refits=_gm_final_prediction(N,S,N,S,Y,blocks,q,m)
    parent={'audit':{'history':history,'final_nuisance_refits':refits}}
    models={};audits={}
    for screen in [False,True]:
        decisions=_cs_screen(history,Y,blocks,screen)
        sfits=_cs_screened_refits(design,Y,parent,decisions)
        name=mode+('_S1' if screen else '_S0')
        fit=_gms_background_predictions(design,Y,sfits,[name])[name]
        # The S0/S1 variants share ALL three source updates. Only the final
        # branch guard/refit differs, exactly as in section20 B000/B001.
        models[name]=fit
        audits[name]={'q':decisions['q'],'m':decisions['m'],
            'q_coefficients':np.asarray([s['q_sources'] for s in sfits]).tolist(),
            'm_coefficients':np.asarray([s['m_sources'] for s in sfits]).tolist(),
            'q_prior':history[-1]['q_prior'],'m_prior':history[-1]['m_prior']}
    return models,audits

def cx_effect(fit):
    return fit['observed_prediction']-fit['observed_zero_prediction']

def cx_energy(effect,blocks):
    w=_gm_weights(blocks);return float(np.sum(w[:,None]*effect*effect)/(4*w.sum()))

def cx_block_error(error,blocks):
    _,ii=np.unique(blocks,return_inverse=True);G=ii.max()+1
    totals=np.zeros((G,4));np.add.at(totals,ii,error);totals/=np.bincount(ii)[:,None]
    return float(np.mean(totals*totals))

def cx_inject(Y,S,kind,seed_parts):
    # Fixed observed counts form the background. These are artificial additions,
    # not known biological ground truth or new independently sampled tissues.
    rng=cx_rng(*seed_parts);near=S[:,::2]
    if kind=='zero_to_positive':
        truth=(Y==0)*(.30*near)
        addition=(rng.random(Y.shape)<truth).astype(float)
    elif kind=='positive_amplification':
        truth=(Y>0)*(4.*near)
        addition=rng.poisson(truth).astype(float)
    else:raise ValueError(kind)
    return Y+addition,truth,{'added_counts':int(addition.sum()),'affected_cells':int(np.any(addition>0,axis=1).sum()),
        'expected_added_counts':float(truth.sum()),'seed_parts':list(map(str,seed_parts)),
        'counts_sha256':hashlib.sha256(np.asarray(Y+addition,dtype='<f8').tobytes()).hexdigest()}

def cx_normal_case(rho,pattern,alpha,rep,seed):
    R=np.eye(8)
    for k in range(4):R[2*k,2*k+1]=R[2*k+1,2*k]=rho
    I=25.*R;L=np.linalg.cholesky(I);beta=np.zeros((4,8))
    for g in range(4):
        if pattern!='null':beta[g,2*g]=alpha
        if pattern=='paired_same':beta[g,2*g+1]=alpha
        if pattern=='paired_opposite':beta[g,2*g+1]=-alpha
    # Common standard noise across rho and amplitude; models exactly paired.
    noise=cx_rng(seed,'normal_score',pattern,rep).normal(size=(4,8))
    U=beta@I+noise@L.T
    summaries=[{'information_I':I,'pseudo_score':U[g],'covariance_model':I} for g in range(4)]
    results={};audits={}
    for mode in ['full','diagonal']:
        results[mode],audits[mode]=cx_prior(summaries,'model',mode)
    results['fixed']=np.array([_gms_profile_fixed(s)[0] for s in summaries])
    rows=[]
    for model,estimate in results.items():
        error=estimate-beta
        contrast=error[:,::2]-error[:,1::2]
        rows.append({'rho':rho,'pattern':pattern,'alpha':alpha,'replicate':rep,'model':model,
            'coefficient_MSE':float(np.mean(error*error)),
            'exposure_effect_MSE':float(np.einsum('gi,ij,gj->',error,R,error)/4),
            'near_far_contrast_MSE':float(np.mean(contrast*contrast)),
            'inactive_coefficient_MSE':float(np.mean(estimate[beta==0]**2)) if (beta==0).any() else None,
            'active_coefficient_MSE':float(np.mean(error[beta!=0]**2)) if (beta!=0).any() else None,
            'pi':audits[model]['pi_slab'] if model in audits else None})
    return rows


## 2. Public fitting API and pilot experiments


In [ ]:
# Public API and portable data access.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits


def load_pilot(root):
    """Load the two frozen four-response pilot designs without external files."""
    chips={}; outcomes={}
    for name in ('C02943C3','A03599E2'):
        with np.load(Path(root)/'data'/(name+'.npz'),allow_pickle=False) as data:
            index=pd.Index(data['index'])
            ch={key:data[key].copy() for key in ('blocks','counts_near','counts_far','xy','component','permutation_counts')}
            ch.update(chip_id=name,index=index,provenance={})
            for table in ('B2','B3'):ch[table]=pd.DataFrame(data[table],index=index,columns=data[table+'_columns'])
            chips[name]=ch;outcomes[name]=data['Y'].copy()
    return chips,outcomes


def make_design(train,test=None,counts=None):
    design=cx_design(train,counts)
    if test is not None:design['T']=cx_design(test)['Z']
    design['folds']=_cd_fold_ids(design['blocks'])[0]
    return design


@threadpool_limits.wrap(limits=1)
def fit_cade(N,S,Y,blocks,screen=False,predict_N=None,predict_S=None):
    """Fit CADE to four readouts and eight sources, with five nuisance columns.

    N includes an intercept followed by four bounded background features.
    S contains target-specific counts transformed as n/(1+n).
    Only training outcomes enter estimation. Prediction inputs may be new rows.
    """
    N=np.asarray(N,float);S=np.asarray(S,float);Y=np.asarray(Y,float)
    blocks=np.asarray(blocks).astype(str)
    if N.shape!=(len(Y),5) or S.shape!=(len(Y),8) or Y.shape[1:]!=(4,) or len(blocks)!=len(Y):
        raise ValueError('Expected N: n x 5, S: n x 8, Y: n x 4, blocks: n.')
    if not np.all(N[:,0]==1) or any(not np.isfinite(x).all() for x in (N,S,Y)) or np.any(Y<0):
        raise ValueError('Finite features, nonnegative outcomes and an intercept are required.')
    if np.any(S<0) or np.any(S>1):raise ValueError('Source exposures must be bounded in [0,1].')
    Tn=N if predict_N is None else np.asarray(predict_N,float)
    Ts=S if predict_S is None else np.asarray(predict_S,float)
    design={'Z':np.column_stack([N,S]),'T':np.column_stack([Tn,Ts]),
        'blocks':blocks,'source':np.r_[np.zeros(5,bool),np.ones(8,bool)]}
    # Source fitting always uses training rows. Reuse final coefficients on T.
    models,audits=cx_hurdle(design,Y,'full')
    key='full_S1' if screen else 'full_S0'
    return models[key]


def fit_paper_models(design,Y):
    """Main estimator, optional screen and the retained published-result controls."""
    Z,T,blocks,source=design['Z'],design['T'],design['blocks'],design['source']
    models,_=cx_hurdle(design,Y,'full')
    out={'cs_B000':models['full_S0'],'cs_B001':models['full_S1']}
    pack=_js_base_pack(Z,Y,blocks,source)
    out.update(fit_cs_standard_eb(design,Y,pack))
    packs,b,cov,support=pack;ind=[]
    for j in range(2):ind.append(_js_uni_prior(b[:,j],cov[:,j,j],support[:,j])[0])
    fields=BA_FIELDS
    for name in ('logistic_fixed_both','spike_independent_plugin','hybrid_independent_q_soft05_m'):
        out[name]={field:np.zeros((len(T),4)) for field in fields}
    for g,item in enumerate(packs):
        q,m,keep=item['q'],item['m'],item['keep'];ix=slice(g*8,(g+1)*8)
        qi=_js_refit(q,Z,keep.astype(float),blocks,source,ind[0][ix])
        mi=_js_refit(m,Z[keep],Y[keep,g],blocks[keep],source,ind[1][ix])
        soft,_=userpip_coefficients(b[ix,1],cov[ix,1,1],support[ix,1])
        ms=_js_refit(m,Z[keep],Y[keep,g],blocks[keep],source,soft['userpip_soft_05'])
        qf=_ef_shrink(q,Z,keep.astype(float),blocks,source,'fixed')
        mf=_ef_shrink(m,Z[keep],Y[keep,g],blocks[keep],source,'fixed')
        for name,(qq,mm) in {'logistic_fixed_both':(qf,mf),'spike_independent_plugin':(qi,mi),'hybrid_independent_q_soft05_m':(qi,ms)}.items():
            for field,value in _js_prediction(qq,mm,T,source).items():out[name][field][:,g]=value
    constant=np.broadcast_to(Y.mean(0),(len(T),4)).copy()
    out['B0_mean']={'observed_prediction':constant,'observed_zero_prediction':constant.copy(),'positive_prediction':None,'positive_zero_prediction':None}
    poisson=np.zeros_like(constant);zero=np.zeros_like(constant)
    for g in range(4):poisson[:,g],zero[:,g],_=_cd_extended_glm(design,Y[:,g],'poisson')
    out['bounded_PoissonCV']={'observed_prediction':poisson,'observed_zero_prediction':zero,'positive_prediction':None,'positive_zero_prediction':None}
    return out


def run_normal(root,replicates=100):
    """Recompute the frozen 40-condition Gaussian-score experiment."""
    rows=[]
    for rho in (0.,.3,.7,.95):
        cases=[('null',0.)]+[(pattern,a) for pattern in ('single','paired_same','paired_opposite') for a in (.2,.6,1.)]
        for pattern,a in cases:
            for rep in range(replicates):rows.extend(cx_normal_case(rho,pattern,a,rep,20261030))
        print('Completed score correlation',rho,flush=True)
    dest=Path(root)/'reruns';dest.mkdir(exist_ok=True)
    pd.DataFrame(rows).to_csv(dest/'normal_scores.csv',index=False)
    return pd.DataFrame(rows)


def run_spatial(root,replicates=12,limit=None):
    """Recompute 32 factorial and 11 stress settings in both prediction directions."""
    root=Path(root);chips,Yreal=load_pilot(root)
    protocol=json.loads((root/'protocols/benchmark.json').read_text());rows=[]
    cases=[('factorial',layout,p,aq,am,None) for layout in protocol['layouts'] for p in protocol['p_grid'] for aq in protocol['q_grid'] for am in protocol['m_grid']]
    cases += [('stress','original',.08,0.,0.,m) for m in protocol['stress_mechanisms']]
    if limit is not None:cases=cases[:limit]
    for stage,layout,p,aq,am,mechanism in cases:
        for rep in range(replicates):
            ch={name:broad_layout(c,layout,rep,protocol,None)[0] for name,c in chips.items()}
            data={}
            for name,c in ch.items():
                if stage=='factorial':data[name]=gm_factorial_generate(c,p,aq,am,rep,protocol,layout)
                elif mechanism in ('block_correlated','nonlinear_saturation'):data[name]=gm_new_stress_generate(c,rep,protocol,mechanism)
                else:data[name]=js_generate(c,.08,1.,rep,protocol,mechanism)
            for train,test in [('C02943C3','A03599E2'),('A03599E2','C02943C3')]:
                fits=fit_paper_models(make_design(ch[train],ch[test]),data[train][0]['Y'])
                info={'stage':stage,'layout':layout,'mechanism':mechanism or f'q{aq}_m{am}','p':p,'aq':aq,'am':am,'replicate':rep,'train':train,'test':test}
                for model,fit in fits.items():rows.append(info|{'model':model}|ef_score(ch[test],*data[test],fit))
        print('Completed',stage,layout,p,aq,am,mechanism,flush=True)
        dest=root/'reruns';dest.mkdir(exist_ok=True);pd.DataFrame(rows).to_csv(dest/'spatial_scores.csv',index=False)
    return pd.DataFrame(rows)


def run_permutations(root,permutations=99):
    """Refit null permutations and paired count injections; keep model failures."""
    root=Path(root);chips,Y=load_pilot(root);rows=[]
    for chip,ch in chips.items():
        for perm in range(1,permutations+1):
            design=make_design(ch,counts=ch['permutation_counts'][perm-1]);base={}
            stages=['null']+(['zero_to_positive','positive_amplification'] if perm>=50 else [])
            for stage in stages:
                current,truth=(Y[chip],np.zeros_like(Y[chip])) if stage=='null' else cx_inject(Y[chip],design['Z'][:,5:],stage,[20261030,'real_spike',chip,perm,stage])[:2]
                for mode in ('full','diagonal'):
                    try:fits,audits=cx_hurdle(design,current,mode)
                    except (FloatingPointError,np.linalg.LinAlgError) as error:
                        for suffix in ('S0','S1'):rows.append({'chip':chip,'permutation':perm,'stage':stage,'model':mode+'_'+suffix,'status':'numerical_failure','failure':str(error)})
                        continue
                    for model,fit in fits.items():
                        effect=cx_effect(fit)
                        if stage=='null':base[model]=effect
                        recovery=effect-base[model]-truth
                        rows.append({'chip':chip,'permutation':perm,'split':'calibration' if perm<=49 else 'evaluation','stage':stage,'model':model,'status':'ok','statistic_effect_energy':cx_energy(effect,design['blocks']),'paired_injection_block_MSE':cx_block_error(recovery,design['blocks']) if stage!='null' else np.nan,'zero_estimator_block_MSE':cx_block_error(truth,design['blocks']) if stage!='null' else np.nan})
            print('Completed permutation',chip,perm,flush=True)
            dest=root/'reruns';dest.mkdir(exist_ok=True);pd.DataFrame(rows).to_csv(dest/'real_scores.csv',index=False)
    return pd.DataFrame(rows)


## 3. Official SuSiE adapter and external comparisons


In [ ]:
# Locked external comparison. Model fitting remains in this notebook.
import os, subprocess, atexit, platform
from datetime import datetime, timezone
from scipy.linalg import cho_solve, cholesky

EX_R_WORKER = r'''
args <- commandArgs(trailingOnly=TRUE)
.libPaths(c(args[1], .libPaths()))
suppressPackageStartupMessages(library(susieR))
suppressPackageStartupMessages(library(jsonlite))
stopifnot(as.character(packageVersion("susieR")) == "0.14.2")
input <- file("stdin", "r")
repeat {
  line <- readLines(input, n=1, warn=FALSE)
  if (length(line)==0) break
  result <- tryCatch({
    request <- fromJSON(line, simplifyVector=FALSE)
    answers <- lapply(request$groups, function(g) {
      if (is.null(g)) return(list(beta=rep(0,8),pip=rep(0,8),converged=TRUE,niter=0,warnings=list()))
      H <- do.call(rbind, lapply(g$H, unlist))
      h <- unlist(g$h); notes <- character()
      fit <- withCallingHandlers(susie_suff_stat(
        XtX=H, Xty=h, yty=g$yty, n=g$n,
        X_colmeans=rep(0,8), y_mean=0, L=8,
        standardize=FALSE, residual_variance=1,
        estimate_residual_variance=FALSE, estimate_prior_variance=TRUE,
        estimate_prior_method="optim", scaled_prior_variance=0.2,
        check_null_threshold=0, max_iter=1000, tol=1e-5,
        refine=FALSE, check_input=TRUE),
        warning=function(w) {notes <<- c(notes,conditionMessage(w)); invokeRestart("muffleWarning")})
      list(beta=as.numeric(colSums(fit$alpha*fit$mu)),
        pip=as.numeric(susie_get_pip(fit)), converged=isTRUE(fit$converged),
        niter=fit$niter, warnings=as.list(unique(notes)))
    })
    list(ok=TRUE, answers=answers)
  }, error=function(e) list(ok=FALSE,error=conditionMessage(e)))
  cat(toJSON(result,auto_unbox=TRUE,digits=16,null="null",na="null"),"\n",sep="")
  flush(stdout())
}
'''

class ExternalSuSiE:
    """Persistent bridge to the unmodified official susieR package."""
    def __init__(self, root):
        root=Path(root); runtime=root/'runtime';runtime.mkdir(exist_ok=True)
        script=runtime/'susie_worker.R';script.write_text(EX_R_WORKER)
        library=os.environ.get('CADE_R_LIBRARY',str(root/'Rlib'))
        self.stderr=(runtime/f'susie_{os.getpid()}.log').open('w')
        self.process=subprocess.Popen([os.environ.get('RSCRIPT','Rscript'),str(script),library],
            stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=self.stderr,text=True,bufsize=1)
        atexit.register(self.close)
    def close(self):
        if self.process.poll() is None:
            self.process.stdin.close()
            try:self.process.wait(timeout=3)
            except subprocess.TimeoutExpired:self.process.terminate()
        self.stderr.close()
    def fit(self,summaries):
        groups=[]
        for summary in summaries:
            group=_gm_likelihood_group(summary,'model')
            if group is None:groups.append(None);continue
            A,u,C=group['A'],group['u'],group['V']
            solved=cho_solve((cholesky(C,lower=True),True),np.column_stack([A,u]))
            H=A.T@solved[:,:8];H=(H+H.T)/2;h=A.T@solved[:,8]
            yty=max(float(u@solved[:,8]),1e-20)
            groups.append({'H':H.tolist(),'h':h.tolist(),'yty':yty,'n':len(u)+1})
        self.process.stdin.write(json.dumps({'groups':groups},allow_nan=False)+'\n');self.process.stdin.flush()
        line=self.process.stdout.readline()
        if not line:raise RuntimeError('Official SuSiE worker terminated; inspect runtime log')
        response=json.loads(line)
        if not response['ok']:raise RuntimeError('Official SuSiE: '+response['error'])
        answers=response['answers']
        return np.array([a['beta'] for a in answers]),answers

def ex_susie_hurdle(design,Y,bridge):
    """Replace only the source posterior; keep the frozen CADE fitting shell."""
    Z=design['Z'];N,S=Z[:,:5],Z[:,5:];blocks=design['blocks'];labels=np.unique(blocks)
    q=np.zeros((4,8));m=np.zeros_like(q);diagnostics=[]
    for iteration in range(3):
        summaries=[]
        for branch in ('q','m'):
            for g in range(4):
                keep=Y[:,g]>0
                if branch=='q':item=_gm_profile(N,S,keep.astype(float),blocks,q[g],'detection',labels)
                else:item=_gm_profile(N[keep],S[keep],Y[keep,g],blocks[keep],m[g],'positive',labels)
                summaries.append(item)
        coef,audit=bridge.fit(summaries);q,m=coef[:4],coef[4:]
        diagnostics.append(audit)
    _,refits=_gm_final_prediction(N,S,N,S,Y,blocks,q,m)
    name='SuSiE_score';fit=_gms_background_predictions(design,Y,refits,[name])[name]
    fit['audit']['susie_iterations']=diagnostics
    return fit

def ex_jsonable(value):
    if isinstance(value,np.ndarray):return value.tolist()
    if isinstance(value,np.generic):return value.item()
    if isinstance(value,dict):return {k:ex_jsonable(v) for k,v in value.items()}
    if isinstance(value,(tuple,list)):return [ex_jsonable(v) for v in value]
    return value

def ex_normal_data(rho,pattern,alpha,rep):
    R=np.eye(8)
    for k in range(4):R[2*k,2*k+1]=R[2*k+1,2*k]=rho
    I=25*R;beta=np.zeros((4,8))
    for g in range(4):
        if pattern!='null':beta[g,2*g]=alpha
        if pattern=='paired_same':beta[g,2*g+1]=alpha
        if pattern=='paired_opposite':beta[g,2*g+1]=-alpha
    noise=cx_rng(20261030,'normal_score',pattern,rep).normal(size=(4,8))
    U=beta@I+noise@np.linalg.cholesky(I).T
    return R,beta,[{'information_I':I,'pseudo_score':U[g],'covariance_model':I} for g in range(4)]

def ex_run_normal(root,shard=0,shards=1):
    root=Path(root);dest=root/'results/external';bridge=ExternalSuSiE(root)
    path=dest/f'normal_{shard}.jsonl';done=set()
    if path.exists():
        for line in path.read_text().splitlines():
            r=json.loads(line);done.add((r['rho'],r['pattern'],r['alpha'],r['replicate']))
    with path.open('a') as f:
        for rho in (0.,.3,.7,.95):
            for pattern,a in [('null',0.)]+[(t,a) for t in ('single','paired_same','paired_opposite') for a in (.2,.6,1.)]:
                for rep in range(shard,100,shards):
                    if (rho,pattern,a,rep) in done:continue
                    row={'rho':rho,'pattern':pattern,'alpha':a,'replicate':rep,'model':'SuSiE_score'}
                    try:
                        R,beta,summary=ex_normal_data(rho,pattern,a,rep)
                        estimate,audit=bridge.fit(summary);error=estimate-beta
                        row.update(status='ok',coefficient_MSE=float(np.mean(error**2)),
                            exposure_effect_MSE=float(np.einsum('gi,ij,gj->',error,R,error)/4),
                            near_far_contrast_MSE=float(np.mean((error[:,::2]-error[:,1::2])**2)),
                            inactive_coefficient_MSE=float(np.mean(estimate[beta==0]**2)),
                            active_coefficient_MSE=float(np.mean(error[beta!=0]**2)) if np.any(beta!=0) else None,
                            all_converged=all(x['converged'] for x in audit),max_iterations=max(x['niter'] for x in audit),
                            warnings=sorted(set(w for x in audit for w in x['warnings'])))
                    except Exception as error:row.update(status='failure',error=str(error))
                    f.write(json.dumps(row)+'\n');f.flush()
            print('NORMAL',shard,rho,flush=True)
    bridge.close()

def ex_run_spatial(root,shard=0,shards=1):
    root=Path(root);dest=root/'results/external';bridge=ExternalSuSiE(root)
    chips,_=load_pilot(root);protocol=json.loads((root/'protocols/benchmark.json').read_text())
    cases=[('factorial',layout,p,aq,am,None) for layout in protocol['layouts'] for p in protocol['p_grid'] for aq in protocol['q_grid'] for am in protocol['m_grid']]
    cases += [('stress','original',.08,0.,0.,m) for m in protocol['stress_mechanisms']]
    path=dest/f'spatial_{shard}.jsonl';done=set()
    if path.exists():
        for line in path.read_text().splitlines():
            r=json.loads(line);done.add((r['case'],r['replicate'],r['train']))
    with path.open('a') as f:
        for ci,(stage,layout,p,aq,am,mechanism) in enumerate(cases):
            for rep in range(shard,12,shards):
                ch={name:broad_layout(c,layout,rep,protocol,None)[0] for name,c in chips.items()};data={}
                for name,c in ch.items():
                    if stage=='factorial':data[name]=gm_factorial_generate(c,p,aq,am,rep,protocol,layout)
                    elif mechanism in ('block_correlated','nonlinear_saturation'):data[name]=gm_new_stress_generate(c,rep,protocol,mechanism)
                    else:data[name]=js_generate(c,.08,1.,rep,protocol,mechanism)
                for train,test in [('C02943C3','A03599E2'),('A03599E2','C02943C3')]:
                    if (ci,rep,train) in done:continue
                    row={'case':ci,'stage':stage,'layout':layout,'p':p,'aq':aq,'am':am,'mechanism':mechanism or f'q{aq}_m{am}','replicate':rep,'train':train,'test':test,'model':'SuSiE_score'}
                    try:
                        fit=ex_susie_hurdle(make_design(ch[train],ch[test]),data[train][0]['Y'],bridge)
                        diag=[a for step in fit['audit']['susie_iterations'] for a in step]
                        row.update(status='ok',**ef_score(ch[test],*data[test],fit),
                            all_converged=all(a['converged'] for a in diag),max_iterations=max(a['niter'] for a in diag),
                            warnings=sorted(set(w for a in diag for w in a['warnings'])))
                    except Exception as error:row.update(status='failure',error=str(error))
                    f.write(json.dumps(ex_jsonable(row))+'\n');f.flush()
            print('SPATIAL',shard,ci,stage,layout,mechanism,flush=True)
    bridge.close()


def ex_native_data(root):
    result={}
    for chip in ('C02943C3','A03599E2'):
        with np.load(Path(root)/'data/locked'/f'{chip}.npz',allow_pickle=False) as z:
            result[chip]={k:z[k].copy() for k in z.files}
    return result

def ex_native_design(ch,panel,geometry='nested',permutation=0,keep=None):
    counts=ch['counts_'+panel][permutation].astype(float).copy()
    if geometry=='nested':counts[:,1::2]+=counts[:,::2]
    elif geometry!='annular':raise ValueError(geometry)
    Z=np.column_stack([ch['N'],counts/(1+counts)])
    keep=np.ones(len(Z),bool) if keep is None else keep
    return {'Z':Z[keep],'T':Z.copy(),'blocks':ch['blocks'][keep],
        'source':np.r_[np.zeros(5,bool),np.ones(8,bool)],'folds':_cd_fold_ids(ch['blocks'][keep])[0]}

def ex_fit_native(design,Y,model,bridge):
    if model=='CADE':return cx_hurdle(design,Y,'full')[0]['full_S0']
    if model=='diagonal':return cx_hurdle(design,Y,'diagonal')[0]['diagonal_S0']
    if model=='SuSiE_score':return ex_susie_hurdle(design,Y,bridge)
    if model=='logistic_hurdle':return fit_cs_standard_eb(design,Y)['logistic_hurdle']
    raise ValueError(model)

def ex_parameters(fit,model):
    rows=fit['audit']['responses'];q=[];m=[];qc=[];mc=[]
    for row in rows:
        if model=='logistic_hurdle':
            q.append(row['q_base_coefficients']);m.append(row['m_base_coefficients'])
            qc.append(row['q_constant']);mc.append(row['m_constant'])
        else:
            q.append(row['q_nuisance_beta']+row['q_sources']);m.append(row['m_nuisance_beta']+row['m_sources'])
            qc.append(None);mc.append(None)
    return {'q':q,'m':m,'q_constant':qc,'m_constant':mc}

def ex_predict(parameters,Z):
    q=expit(Z@np.asarray(parameters['q']).T);m=np.maximum(Z@np.asarray(parameters['m']).T,0)
    for g in range(4):
        if parameters['q_constant'][g] is not None:q[:,g]=parameters['q_constant'][g]
        if parameters['m_constant'][g] is not None:m[:,g]=parameters['m_constant'][g]
    return q*m

def ex_effect(parameters,Z,target=None):
    zero=Z.copy()
    if target is None:zero[:,5:]=0
    else:zero[:,5+2*target:7+2*target]=0
    return ex_predict(parameters,Z)-ex_predict(parameters,zero)

def ex_run_native(root,shard=0,shards=1):
    """Fit all locked native cases, preserving errors and coefficients."""
    root=Path(root);data=ex_native_data(root);bridge=ExternalSuSiE(root)
    spec=json.loads((root/'protocols/external/locked_protocol.json').read_text())['native']
    path=root/'results/external'/f'native_{shard}.jsonl';done=set()
    if path.exists():
        for line in path.read_text().splitlines():
            r=json.loads(line);done.add((r['chip'],r['panel'],r['geometry'],r['kind'],r['repeat'],r['model']))
    jobs=[]
    for chip,ch in data.items():
        for panel in spec['panels']:
            for geometry in ('nested','annular'):
                jobs.append((chip,panel,geometry,'full',0))
                if geometry=='nested':
                    jobs.extend((chip,panel,geometry,'permutation',p) for p in range(1,100))
                    jobs.extend((chip,panel,geometry,'delete_group',p) for p in range(5))
    with path.open('a') as out:
        for ji,(chip,panel,geometry,kind,repeat) in enumerate(jobs):
            if ji%shards!=shard:continue
            ch=data[chip];keep=np.ones(len(ch['Y']),bool)
            if kind=='delete_group':
                labels=sorted(np.unique(ch['blocks']),key=lambda b:hashlib.sha256(('CADE-native-lock-20261030-v1|'+b).encode()).hexdigest())
                excluded=set(labels[repeat::5]);keep=np.array([b not in excluded for b in ch['blocks']])
            d=ex_native_design(ch,panel,geometry,repeat if kind=='permutation' else 0,keep)
            for model in spec['models']:
                key=(chip,panel,geometry,kind,repeat,model)
                if key in done:continue
                row=dict(zip(['chip','panel','geometry','kind','repeat','model'],key))
                try:
                    fit=ex_fit_native(d,ch['Y'][keep],model,bridge);par=ex_parameters(fit,model)
                    assert np.allclose(ex_predict(par,d['T']),fit['observed_prediction'],rtol=1e-8,atol=1e-10)
                    effect=ex_effect(par,d['T']);w=_gm_weights(ch['blocks']);w/=w.sum()
                    target_effects=[ex_effect(par,d['T'],k) for k in range(4)]
                    row.update(status='ok',parameters=par,effect_energy=float(np.mean(w@(effect**2))),
                        target_gene_energy=[(w@(e**2)).tolist() for e in target_effects],
                        target_gene_signed_mean=[(w@e).tolist() for e in target_effects],
                        nonzero_source_coefficients=int(np.count_nonzero(np.abs(np.r_[np.asarray(par['q'])[:,5:].ravel(),np.asarray(par['m'])[:,5:].ravel()])>1e-8)))
                    if model=='SuSiE_score':
                        diag=[a for step in fit['audit']['susie_iterations'] for a in step]
                        row.update(all_converged=all(a['converged'] for a in diag),max_iterations=max(a['niter'] for a in diag),warnings=sorted(set(w for a in diag for w in a['warnings'])))
                except Exception as error:row.update(status='failure',error=str(error))
                out.write(json.dumps(ex_jsonable(row))+'\n');out.flush()
            if repeat%10==0 or kind!='permutation':print('NATIVE',shard,chip,panel,geometry,kind,repeat,flush=True)
    bridge.close()


## 4. Matched prior controls and spatial validation


In [ ]:
# Frozen follow-up validation. The published estimator remains unchanged.
import time, sys, resource
from scipy.spatial.distance import cdist

FU_MODELS=('CADE_pooled','CADE_unpooled_same_grid','CADE_MAP_same_prior','SuSiE_score')

def fu_scale(summaries):
    scale=1e-8
    for s in summaries:
        d=np.diag(s['information_I']);v=d>max(float(d.max()),1e-30)*1e-10
        if v.any():scale=max(scale,float(np.max((s['pseudo_score'][v]/d[v])**2)),float(np.max(np.diag(s['covariance_model'])[v]/d[v]**2)))
    return scale

def fu_prior(summaries, grid_multiplier=1.):
    """Exact matched-grid pooling and support-averaging ablations.

    The marginal likelihood is identical to cx_prior. Only the sharing of
    hyperparameters or the final posterior decision rule is changed.
    """
    groups=[_gm_likelihood_group(s,'model') for s in summaries]
    scale=fu_scale(summaries);taus=scale*np.geomspace(1e-3,10.,9)*grid_multiplier
    grid=np.linspace(0,1,41);weights=np.array([_gm_pi_weights(p) for p in grid])
    cache=[]
    for tau in taus:
        components=[_gm_support_components(g,float(tau)) for g in groups]
        ll=np.array([c[0]-c[0][0] for c in components]);cache.append((ll,components))
    def choose(indices):
        best=(-np.inf,None);candidates=[]
        for ti,(ll,components) in enumerate(cache):
            l=ll[indices];values=logsumexp(l[None,:,:]+weights[:,None,:],axis=2).sum(1)
            ix=int(np.argmax(values));pi=float(grid[ix]);value=float(values[ix])
            obj=lambda p:-float(logsumexp(l+_gm_pi_weights(float(p)),axis=1).sum())
            opt=minimize_scalar(obj,bounds=(grid[max(ix-1,0)],grid[min(ix+1,40)]),method='bounded',options={'xatol':1e-8})
            if opt.success and -opt.fun>value+1e-10:pi,value=float(opt.x),float(-opt.fun)
            candidates.append({'tau2':float(taus[ti]),'pi':pi,'log_marginal_BF_vs_null':value})
            if value>best[0]+1e-10:best=(value,(pi,ti))
        pi,ti=best[1];means=[];maps=[];pips=[]
        for gi in indices:
            ll,components=cache[ti];lw=ll[gi]+_gm_pi_weights(pi)
            resp=np.exp(lw-logsumexp(lw));component=components[gi][1]
            means.append(resp@component if pi else np.zeros(8))
            maps.append(component[int(np.argmax(lw))] if pi else np.zeros(8))
            pips.append(resp@GM_SCORE_STATES if groups[gi] is not None else np.zeros(8))
        audit={'pi_slab':pi,'tau2':float(taus[ti]) if pi else 0.,'scale_grid':scale,
               'grid_candidates':candidates,'max_logBF':float(best[0]),'posterior_inclusion':np.asarray(pips).tolist()}
        return np.asarray(means),np.asarray(maps),audit
    pooled,modal,pa=choose(list(range(len(groups))));unpooled=[];ua=[]
    for g in range(len(groups)):
        a,_,b=choose([g]);unpooled.append(a[0]);ua.append(b)
    return {'CADE_pooled':pooled,'CADE_unpooled_same_grid':np.asarray(unpooled),
            'CADE_MAP_same_prior':modal},{'pooled':pa,'unpooled':ua}

def fu_fit(design,Y,model,bridge=None,iterations=3,grid_multiplier=1.,background='stable'):
    """Same fitting shell for pooling, posterior decision and SuSiE controls."""
    N,S=design['Z'][:,:5],design['Z'][:,5:];blocks=design['blocks'];labels=np.unique(blocks)
    q=np.zeros((4,8));m=np.zeros_like(q);history=[]
    for it in range(iterations):
        qs=[];ms=[]
        for g in range(4):
            keep=Y[:,g]>0
            qs.append(_gm_profile(N,S,keep.astype(float),blocks,q[g],'detection',labels))
            ms.append(_gm_profile(N[keep],S[keep],Y[keep,g],blocks[keep],m[g],'positive',labels))
        if model=='SuSiE_score':
            coef,audit=bridge.fit(qs+ms);q,m=coef[:4],coef[4:]
            history.append({'converged':all(a['converged'] for a in audit),'maxiter':max(a['niter'] for a in audit)})
        elif model=='background_only':q[:]=0;m[:]=0
        else:
            qr,qa=fu_prior(qs,grid_multiplier);mr,ma=fu_prior(ms,grid_multiplier)
            q,m=qr[model],mr[model];history.append({'q_prior':qa['pooled'],'m_prior':ma['pooled']})
    Tn,Ts=design['T'][:,:5],design['T'][:,5:]
    raw,refits=_gm_final_prediction(N,S,Tn,Ts,Y,blocks,q,m)
    if background=='raw':out=raw
    else:out=_gms_background_predictions(design,Y,refits,[model])[model]
    out['followup']={'q_sources':q.tolist(),'m_sources':m.tolist(),'history':history,'iterations':iterations}
    return out

def fu_protocol(root):return json.loads((Path(root)/'protocols/followup/protocol.json').read_text())

def fu_normal_data(protocol,rho,pattern,rep):
    rng=cx_rng(protocol['seed'],'normal',pattern,rep)
    R=np.eye(8)
    for k in range(4):R[2*k,2*k+1]=R[2*k+1,2*k]=rho
    beta=np.zeros((4,8));info=np.asarray(protocol['normal']['information_scale']);a=protocol['normal']['effect_scale']
    for g in range(4):
        if pattern=='homogeneous':beta[g,2*g]=.6
        elif pattern=='heterogeneous':
            beta[g,2*g]=a[g];beta[g,2*g+1]=(-1 if g%2 else 1)*a[g]*.7
        elif pattern=='mixed_density':
            active=[[],[2],[4,5],[0,2,5,7]][g];beta[g,active]=np.resize(np.array([1,-1]),len(active))*.5
        elif pattern!='null':raise ValueError(pattern)
    summaries=[]
    for g in range(4):
        I=info[g]*R;U=I@beta[g]+np.linalg.cholesky(I)@rng.normal(size=8)
        summaries.append({'information_I':I,'pseudo_score':U,'covariance_model':I})
    return R,beta,summaries

def fu_write_row(handle,row):
    handle.write(json.dumps(ex_jsonable(row),allow_nan=False)+'\n');handle.flush()

def fu_run_normal(root,shard=0,shards=1):
    root=Path(root);p=fu_protocol(root);dest=root/'results/followup';dest.mkdir(exist_ok=True)
    path=dest/f'normal_{shard}.jsonl';done=set()
    done={tuple(json.loads(l)[k] for k in ('rho','pattern','replicate')) for file in dest.glob('normal_*.jsonl') for l in file.read_text().splitlines()}
    bridge=ExternalSuSiE(root)
    with path.open('a') as out:
        for rho in p['normal']['rho']:
            for pattern in p['normal']['patterns']:
                for rep in range(shard,p['normal']['replicates'],shards):
                    if (rho,pattern,rep) in done:continue
                    row={'rho':rho,'pattern':pattern,'replicate':rep};start=time.perf_counter()
                    try:
                        R,beta,s=fu_normal_data(p,rho,pattern,rep);t=time.perf_counter();est,audit=fu_prior(s)
                        exact_seconds=time.perf_counter()-t;t=time.perf_counter();est['SuSiE_score'],sa=bridge.fit(s)
                        susie_seconds=time.perf_counter()-t;metrics={}
                        for name,b in est.items():
                            e=b-beta;metrics[name]={'combined_effect_MSE':float(np.einsum('gi,ij,gj->',e,R,e)/4),
                                'coefficient_MSE':float(np.mean(e*e)),'inactive_MSE':float(np.mean(b[beta==0]**2))}
                        row.update(status='ok',metrics=metrics,pooled_pi=audit['pooled']['pi_slab'],
                            unpooled_pi=[a['pi_slab'] for a in audit['unpooled']],
                            exact_all_ablations_seconds=exact_seconds,susie_seconds=susie_seconds,
                            susie_converged=all(a['converged'] for a in sa))
                    except Exception as e:row.update(status='failure',error=repr(e))
                    row['seconds']=time.perf_counter()-start;fu_write_row(out,row)
                print('FOLLOWUP NORMAL',shard,rho,pattern,flush=True)
    bridge.close()

def fu_spatial_data(protocol,encoding,mechanism,rep,split):
    rng=cx_rng(protocol['seed'],'new-spatial',encoding,mechanism,rep,split)
    spec=protocol['spatial'];n=spec[split+'_receivers'];xy=rng.uniform(0,1600,(n,2))
    # Half the sources form local clusters, independently in train and test.
    sources=[]
    for k in range(4):
        centers=rng.uniform(150,1450,(2,2));src=np.vstack([rng.uniform(0,1600,(30,2)),
            centers[np.arange(30)%2]+rng.normal(0,90,(30,2))]);sources.append(src)
    counts=[]
    for src in sources:
        d=cdist(xy,src)
        if encoding=='annuli':a=((d>20)&(d<=50)).sum(1);b=((d>50)&(d<=100)).sum(1)
        elif encoding=='nested':a=((d>20)&(d<=50)).sum(1);b=((d>20)&(d<=100)).sum(1)
        elif encoding=='exponential':a=(np.exp(-d/35)*(d>20)*(d<=200)).sum(1);b=(np.exp(-d/80)*(d>20)*(d<=200)).sum(1)
        else:raise ValueError(encoding)
        counts.extend([a,b])
    counts=np.asarray(counts).T;S=counts/(1+counts)
    pooled=counts.reshape(n,4,2).sum(1);pooled=pooled/(1+pooled)
    tree=cKDTree(xy);density=np.column_stack([tree.query_ball_point(xy,r,return_length=True)-1 for r in (50,100)])
    N=np.column_stack([np.ones(n),pooled,density/(1+density)])
    blocks=np.array([f'{split}:{i}:{j}' for i,j in np.floor(xy/200).astype(int)])
    qcoef=np.zeros((4,8));mcoef=np.zeros_like(qcoef)
    for g in range(4):qcoef[g,2*g]=1.1;mcoef[g,2*g]=.8
    if mechanism=='null_low':qcoef[:]=0;mcoef[:]=0
    if mechanism=='heterogeneous':
        qcoef=np.zeros_like(qcoef);mcoef=np.zeros_like(mcoef)
        qcoef[1,2:4]=[1.1,-.5];mcoef[2,4:6]=[1.8,.4]
        qcoef[3,[0,4,7]]=[1.4,-.8,.5];mcoef[3,[0,4,7]]=[-.7,.9,.4]
    baseq=logit(np.asarray(spec['detection_base']))[None,:]+.3*N[:,1:2]-.15*N[:,2:3]
    basem=np.broadcast_to(np.array([1.3,2.,3.,4.])[None,:],(n,4)).copy()+.25*N[:,2:3]
    if mechanism=='spatial_residual':
        centers=rng.uniform(0,1600,(20,2));phi=np.exp(-cdist(xy,centers)**2/(2*250**2))
        field=phi@rng.normal(size=(20,4));field-=field.mean(0);field/=np.maximum(field.std(0),1e-8)
        baseq=baseq+.55*field;basem=basem*np.exp(.18*field)
    q0=expit(baseq);m0=basem;q=expit(baseq+S@qcoef.T);m=basem+S@mcoef.T
    assert np.all(m>0)
    Y=rng.gamma(4.,m/4.)*(rng.random((n,4))<q)
    design={'Z':np.column_stack([N,S]),'T':np.column_stack([N,S]),'blocks':blocks,
        'source':np.r_[np.zeros(5,bool),np.ones(8,bool)]}
    truth={'mean':q*m,'effect':q*m-q0*m0,'qcoef':qcoef,'mcoef':mcoef}
    return design,Y,truth

def fu_spatial_metrics(fit,truth,blocks):
    e=cx_effect(fit)-truth['effect'];m=fit['observed_prediction']-truth['mean'];w=_gm_weights(blocks)
    return {'effect_MSE':float(np.sum(w[:,None]*e*e)/(4*w.sum())),
        'mean_MSE':float(np.sum(w[:,None]*m*m)/(4*w.sum())),
        'block_effect_MSE':cx_block_error(e,blocks),'block_mean_MSE':cx_block_error(m,blocks)}

def fu_run_spatial(root,shard=0,shards=1):
    root=Path(root);p=fu_protocol(root);dest=root/'results/followup';dest.mkdir(exist_ok=True)
    path=dest/f'spatial_{shard}.jsonl';done=set()
    done={tuple(json.loads(l)[k] for k in ('encoding','mechanism','replicate','model','iterations','grid_multiplier','background')) for file in dest.glob('spatial_*.jsonl') for l in file.read_text().splitlines()}
    bridge=ExternalSuSiE(root)
    with path.open('a') as out:
        for encoding in p['spatial']['encodings']:
            for mechanism in p['spatial']['mechanisms']:
                for rep in range(shard,p['spatial']['replicates'],shards):
                    train,Y,truth=fu_spatial_data(p,encoding,mechanism,rep,'train')
                    test,Ytest,ttruth=fu_spatial_data(p,encoding,mechanism,rep,'test');train['T']=test['Z']
                    configs=[(m,3,1.,'stable') for m in FU_MODELS]+[('background_only',1,1.,'stable')]
                    if rep<4 and mechanism!='null_low':
                        configs += [('CADE_pooled',it,1.,'stable') for it in (1,5)]
                        configs += [('CADE_pooled',3,mul,'stable') for mul in (.5,2.)]
                        configs += [('CADE_pooled',3,1.,'raw')]
                    for model,it,mul,bg in configs:
                        key=(encoding,mechanism,rep,model,it,mul,bg)
                        if key in done:continue
                        row=dict(zip(('encoding','mechanism','replicate','model','iterations','grid_multiplier','background'),key));start=time.perf_counter()
                        try:
                            fit=fu_fit(train,Y,model,bridge,it,mul,bg);metrics=fu_spatial_metrics(fit,ttruth,test['blocks'])
                            row.update(status='ok',**metrics,positive_train=(Y>0).sum(0).tolist())
                            if model=='SuSiE_score':row['susie_converged']=all(a['converged'] for a in fit['followup']['history'])
                        except Exception as e:row.update(status='failure',error=repr(e))
                        row['seconds']=time.perf_counter()-start;fu_write_row(out,row)
                    if rep%5==0:print('FOLLOWUP SPATIAL',shard,encoding,mechanism,rep,flush=True)
    bridge.close()


## 5. Held-out specimen preparation and prediction


In [ ]:
# Input-to-result preparation for the untouched third Stereo-seq chip.
import h5py
from scipy.sparse.csgraph import connected_components

def fu_file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()

def fu_prepare_gef(root,path,chip='B03018A2'):
    """Extract frozen readouts, barcodes and geometry from an original GEF.

    Read the raw cell counts in batches. Derive QC totals from cellExp rather
    than relying on a potentially narrow integer total in the cell metadata.
    No target or readout is selected by measured effect or model performance.
    """
    root=Path(root);path=Path(path);p=fu_protocol(root);genes=sum(p['native']['readout_panels'].values(),[])
    if chip=='B03018A2':
        if path.stat().st_size!=p['native']['expected_bytes']:raise ValueError('Third-chip raw size differs from the frozen GEO record')
        if fu_file_hash(path)!='06635972f0a2054951ad9ef220f921a2c0e0b06f319efc2fdfcc2bc5beef5fcc':raise ValueError('Third-chip raw checksum mismatch')
    with h5py.File(path,'r') as f:
        c=f['cellBin/cell'][:];g=f['cellBin/gene'][:];rawnames=np.char.decode(g['geneName'],'utf-8')
        names=np.char.replace(rawnames,'_','-');assert np.array_equal(c['id'],np.arange(len(c)))
        assert all(np.sum(names==x)==1 for x in genes)
        def read_column(j):
            row=g[j];e=f['cellBin/geneExp'][int(row['offset']):int(row['offset'])+int(row['cellCount'])]
            a=np.zeros(len(c),np.int64);np.add.at(a,e['cellID'],e['count']);return a
        ids=np.flatnonzero(np.char.startswith(names,'sgrna-'))
        barcode=np.column_stack([read_column(j) for j in ids]);targets=np.array([n[6:].lower() for n in names[ids]])
        assert len(targets)==18 and len(set(targets))==18
        Y=np.column_stack([read_column(int(np.flatnonzero(names==x)[0])) for x in genes])
        mtids=np.flatnonzero(np.char.startswith(np.char.lower(names),'mt-'));assert len(mtids)>0
        mt=sum((read_column(j) for j in mtids),np.zeros(len(c),np.int64))
        total=np.zeros(len(c),np.int64)
        for lo in range(0,len(c),10000):
            hi=min(lo+10000,len(c));start=int(c['offset'][lo]);end=int(c['offset'][hi]) if hi<len(c) else len(f['cellBin/cellExp'])
            e=f['cellBin/cellExp'][start:end]
            assert len(e)==int(c['geneCount'][lo:hi].astype(np.int64).sum())
            total[lo:hi]=np.bincount(np.repeat(np.arange(hi-lo),c['geneCount'][lo:hi]),weights=e['count'],minlength=hi-lo).astype(np.int64)
        pitch=float(np.asarray(f.attrs['resolution']).ravel()[0])/1000.;assert pitch==.5
        total_match=bool(np.array_equal(total,c['expCount']))
    endogenous=total-barcode.sum(1);nbar=(barcode>0).sum(1);calls=np.full(len(c),'unknown',dtype='<U32')
    calls[nbar>1]='multiple';calls[nbar==1]=targets[barcode[nbar==1].argmax(1)]
    xy=np.column_stack([c['x'],c['y']])*pitch;d,j=cKDTree(xy).query(xy,k=9)
    row=np.repeat(np.arange(len(c)),8);col=j[:,1:].ravel();keep=d[:,1:].ravel()<=100
    graph=sparse.csr_matrix((np.ones(keep.sum()),(row[keep],col[keep])),shape=(len(c),len(c)))
    _,component=connected_components(graph,directed=False);sizes=np.bincount(component)
    order=np.argsort(-sizes,kind='stable');remap=np.empty(len(sizes),int);remap[order]=np.arange(len(sizes));component=remap[component]
    sizes=np.bincount(component);large=sizes[component]>=1000;blocks=np.full(len(c),'',dtype='<U80')
    for co in np.unique(component):
        mask=component==co;ij=np.floor((xy[mask]-xy[mask].min(0))/500).astype(int)
        blocks[mask]=[f'{chip}:c{co}:x{i}:y{j}' for i,j in ij]
    qc=(endogenous>=100)&(endogenous<=6000)&(mt/np.maximum(endogenous,1)<=.2)
    density=qc&large;single=density&(nbar==1);pooled=density&(nbar>0);receiver=density&(nbar==0)
    nearest=np.full(len(c),np.inf);nearest_single=nearest.copy()
    for co in np.unique(component[large]):
        inside=component==co;ix=np.flatnonzero(inside)
        for mask,out in [(pooled,nearest),(single,nearest_single)]:
            ids=np.flatnonzero(mask&inside)
            if len(ids):out[ix]=cKDTree(xy[ids]).query(xy[ix])[0]
    eligible=receiver&(nearest>20)&(nearest_single<=100)
    labels,cnt=np.unique(blocks[eligible],return_counts=True);final=eligible&np.isin(blocks,labels[cnt>=20]);rows=np.flatnonzero(final)
    if not len(rows):raise ValueError('No eligible receivers under frozen QC and geometry')
    source=np.flatnonzero(single);Sxy=xy[source];slabels=calls[source];strata=np.char.add(blocks[source],np.where(barcode[source].sum(1)>=2,'::ge2','::one'))
    near=np.zeros((len(rows),4),int);far=near.copy();Nraw=np.zeros((len(rows),4),int);rr=[];ss=[];dd=[]
    for co in np.unique(component[rows]):
        ri=np.flatnonzero(component[rows]==co);si=np.flatnonzero(component[source]==co)
        R=xy[rows[ri]]
        if len(si):
            edges=cKDTree(R).sparse_distance_matrix(cKDTree(Sxy[si]),100,output_type='coo_matrix')
            rr.extend(ri[edges.row]);ss.extend(si[edges.col]);dd.extend(edges.data)
        for index,mask in [(0,pooled),(2,density)]:
            tree=cKDTree(xy[mask&(component==co)]);n20=tree.query_ball_point(R,20,return_length=True)
            n50=tree.query_ball_point(R,50,return_length=True);n100=tree.query_ball_point(R,100,return_length=True)
            if index==0:Nraw[ri,0]=n50-n20;Nraw[ri,1]=n100-n50
            else:Nraw[ri,2]=n50-1;Nraw[ri,3]=n100-1
    rr=np.asarray(rr,int);ss=np.asarray(ss,int);dd=np.asarray(dd)
    encoding=(slabels[:,None]==np.asarray(p['native']['source_targets'])[None,:]).astype(int);arrays={};mats=[]
    for key,mask in [('near',(dd>20)&(dd<=50)),('far',(dd>50)&(dd<=100))]:
        a=sparse.csr_matrix((np.ones(mask.sum(),np.int8),(rr[mask],ss[mask])),shape=(len(rows),len(source)))
        mats.append(a);arrays.update({key+'_data':a.data,key+'_indices':a.indices,key+'_indptr':a.indptr,key+'_shape':a.shape})
    near,far=[a@encoding for a in mats];counts=np.stack([near,far],axis=2).reshape(len(rows),8)
    N=np.column_stack([np.ones(len(rows)),Nraw/(1+Nraw)])
    out=root/'data/followup';out.mkdir(exist_ok=True);dest=out/(chip+'.npz')
    np.savez_compressed(dest,N=N,counts=counts,Y=Y[rows],genes=np.asarray(genes),blocks=blocks[rows],xy=xy[rows],
        component=component[rows],cell_id=c['id'][rows],source_labels=slabels,source_strata=strata,source_xy=Sxy,
        source_blocks=blocks[source],source_component=component[source],**arrays)
    threshold=p['native']['support_minima'];support=[]
    for k,target in enumerate(p['native']['source_targets']):
        sm=slabels==target;source_blocks=len(np.unique(blocks[source][sm]));exposed=counts[:,2*k:2*k+2].sum(1)>0
        for gi,gene in enumerate(genes):
            positive=Y[rows,gi]>0;pb=len(np.unique(blocks[rows][positive]));pe=int((positive&exposed).sum())
            passed=sm.sum()>=threshold['sources_per_target'] and source_blocks>=threshold['source_blocks'] and positive.sum()>=threshold['positive_readouts'] and pb>=threshold['positive_blocks'] and pe>=threshold['positive_exposed_receivers']
            support.append({'chip':chip,'target':target,'gene':gene,'sources':int(sm.sum()),'source_blocks':source_blocks,
                'positive_receivers':int(positive.sum()),'positive_blocks':pb,'positive_exposed_receivers':pe,
                'detection_rate':float(positive.mean()),'support_pass':bool(passed)})
    audit={'chip':chip,'raw_cells':len(c),'qc_cells':int(qc.sum()),'eligible_receivers':len(rows),
        'receiver_blocks':len(np.unique(blocks[rows])),'single_sources':len(source),'all_source_support':dict(zip(*np.unique(slabels,return_counts=True))),
        'component_sizes':sizes.tolist(),'coordinate_pitch_um':pitch,'stored_total_matches_streamed_total':total_match,
        'raw_sha256':fu_file_hash(path),'raw_bytes':path.stat().st_size,'output_sha256':fu_file_hash(dest),
        'input_path':str(path),'timestamp_utc':datetime.now(timezone.utc).isoformat(),'model_results_inspected':False,
        'sample_scope':'Previously unused chip, same study; no inferred animal IDs.'}
    pd.DataFrame(support).to_csv(root/'results/followup/native_support.csv',index=False)
    (root/'protocols/followup/native_preparation.json').write_text(json.dumps(ex_jsonable(audit),indent=2)+'\n')
    return audit

def fu_native_counts(data,permutation=0):
    labels=data['source_labels'].copy();strata=data['source_strata']
    if permutation:
        rng=cx_rng('CADE-independent-followup-20261030-v1','third-chip-permutation',permutation)
        for s in np.unique(strata):
            ix=np.flatnonzero(strata==s);labels[ix]=rng.permutation(labels[ix])
    encoding=(labels[:,None]==np.array(['clu','fasn','stk39','oligo2'])[None,:]).astype(int);counts=[]
    for key in ('near','far'):
        a=sparse.csr_matrix((data[key+'_data'],data[key+'_indices'],data[key+'_indptr']),shape=tuple(data[key+'_shape']))
        counts.append(a@encoding)
    return np.stack(counts,axis=2).reshape(len(data['N']),8)

def fu_native_design(N,counts,blocks,T=None):
    Z=np.column_stack([N,counts/(1+counts)])
    return {'Z':Z,'T':Z.copy() if T is None else T,'blocks':blocks,
        'source':np.r_[np.zeros(5,bool),np.ones(8,bool)]}

def fu_native_metrics(pred,Y,blocks):
    residual=pred-Y;_,ix=np.unique(blocks,return_inverse=True);B=ix.max()+1
    block=np.zeros((B,4));np.add.at(block,ix,residual);block/=np.bincount(ix)[:,None]
    shift=np.mean(block,axis=0)
    return {'block_MSE':float(np.mean(block**2)),'centered_block_MSE':float(np.mean((block-shift)**2)),
        'mean_shift_squared':float(np.mean(shift**2)),'per_gene_block_MSE':np.mean(block**2,axis=0).tolist()}

def fu_run_native(root):
    root=Path(root);p=fu_protocol(root);dest=root/'results/followup';bridge=ExternalSuSiE(root)
    with np.load(root/'data/followup/B03018A2.npz',allow_pickle=False) as f:test={k:f[k] for k in f.files}
    chips,oldY=load_pilot(root);locked=ex_native_data(root)
    testdesign=fu_native_design(test['N'],test['counts'],test['blocks']);rows=[]
    for panel,genes in p['native']['readout_panels'].items():
        designs={name:make_design(ch) for name,ch in chips.items()}
        Ys=oldY if panel=='pilot' else {name:locked[name]['Y'] for name in chips}
        yidx=[list(test['genes']).index(g) for g in genes];Ytest=test['Y'][:,yidx]
        training_sets=[('combined',list(chips))]+[(name,[name]) for name in chips]
        for label,ids in training_sets:
            d={'Z':np.concatenate([designs[i]['Z'] for i in ids]),'T':testdesign['Z'],
                'blocks':np.concatenate([designs[i]['blocks'] for i in ids]),'source':designs[ids[0]]['source']}
            Y=np.concatenate([Ys[i] for i in ids]);models=list(FU_MODELS)+['background_only','train_mean']
            for model in models:
                row={'panel':panel,'train':label,'test':'B03018A2','model':model};start=time.perf_counter()
                try:
                    if model=='train_mean':pred=np.broadcast_to(Y.mean(0),(len(Ytest),4));zero=pred
                    else:
                        fit=fu_fit(d,Y,model,bridge);pred=fit['observed_prediction'];zero=fit['observed_zero_prediction']
                        if label=='combined':
                            (dest/f'native_parameters_{panel}_{model}.json').write_text(json.dumps(ex_jsonable(fit['audit']),indent=2))
                            np.savez_compressed(dest/f'native_{panel}_{model}.npz',prediction=pred,zero=zero,Y=Ytest,blocks=test['blocks'])
                    row.update(status='ok',source_on=fu_native_metrics(pred,Ytest,test['blocks']),source_zero=fu_native_metrics(zero,Ytest,test['blocks']))
                except Exception as e:row.update(status='failure',error=repr(e))
                row['seconds']=time.perf_counter()-start;rows.append(row);print('NATIVE',panel,label,model,row['status'],flush=True)
    with (dest/'native.jsonl').open('w') as f:
        for row in rows:fu_write_row(f,row)
    bridge.close();return rows


## 6. Restricted source-label randomization


In [ ]:
# Locked test-chip predictive randomization: trained parameters stay fixed.
def fu_run_native_permutations(root):
    root=Path(root);spec=fu_protocol(root)['native'];dest=root/'results/followup'
    with np.load(root/'data/followup/B03018A2.npz',allow_pickle=False) as f:data={k:f[k] for k in f.files}
    assert np.array_equal(fu_native_counts(data),data['counts'])
    strata=data['source_strata'];labels=data['source_labels'];groups=[np.flatnonzero(strata==s) for s in np.unique(strata)]
    movable=np.zeros(len(labels),bool)
    for ix in groups:
        if len(np.unique(labels[ix]))>1:movable[ix]=True
    selected=np.isin(labels,spec['source_targets']);fraction=float(movable[selected].mean()) if selected.any() else 0.
    N=data['N'];blocks=data['blocks'];_,ix=np.unique(blocks,return_inverse=True);sizes=np.bincount(ix);B=len(sizes)
    def blockmean(a):
        ans=np.zeros((B,4));np.add.at(ans,ix,a);return ans/sizes[:,None]
    params={};observed={};zero_errors={};truth={}
    for panel,genes in spec['readout_panels'].items():
        truth[panel]=blockmean(data['Y'][:,[list(data['genes']).index(g) for g in genes]])
        for model in FU_MODELS:
            key=panel+'__'+model
            a=json.loads((dest/f'native_parameters_{panel}_{model}.json').read_text());r=a['responses']
            params[key]=(np.array([x['q_nuisance_beta'] for x in r]),np.array([x['m_nuisance_beta'] for x in r]),np.array([x['q_sources'] for x in r]),np.array([x['m_sources'] for x in r]))
            qb,mb,qs,ms=params[key];zero=expit(N@qb.T)*np.maximum(N@mb.T,0)
            zero_errors[key]=np.mean((blockmean(zero)-truth[panel])**2,axis=0)
    path=dest/'native_predictive_permutations.jsonl';done=set()
    if path.exists():done={json.loads(l)['permutation'] for l in path.read_text().splitlines()}
    hashes=set();start=time.perf_counter()
    with path.open('a') as f:
        for perm in range(spec['permutations']+1):
            counts=fu_native_counts(data,perm);digest=hashlib.sha256(counts.tobytes()).hexdigest();hashes.add(digest)
            if perm in done:continue
            S=counts/(1+counts);statistics={}
            for key,(qb,mb,qs,ms) in params.items():
                panel=key.split('__')[0];pred=expit(N@qb.T+S@qs.T)*np.maximum(N@mb.T+S@ms.T,0)
                per_gene=zero_errors[key]-np.mean((blockmean(pred)-truth[panel])**2,axis=0)
                statistics[key]={'block_MSE_gain':float(per_gene.mean()),'per_gene_gain':per_gene.tolist()}
            fu_write_row(f,{'permutation':perm,'exposure_sha256':digest,'statistics':statistics})
            if perm%100==0:print('PREDICTIVE RANDOMIZATION',perm,flush=True)
    audit={'draws_including_observed':spec['permutations']+1,'unique_exposure_matrices':len(hashes),
        'movable_selected_source_fraction':fraction,'exchangeability_support_pass':fraction>=.5,
        'fit_on_test_outcomes':False,'model_refit_per_permutation':False,'seconds':time.perf_counter()-start,
        'test':'One-sided held-out block-MSE improvement under restricted source-label exchangeability; not a null-effect FPR estimate.'}
    (root/'protocols/followup/predictive_randomization_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
    return audit


## 7. Within-specimen component holdouts


In [ ]:
# Secondary validation frozen after support audit, before inspecting prediction errors.
def fu_run_components(root):
    root=Path(root);p=fu_protocol(root);dest=root/'results/followup';bridge=ExternalSuSiE(root)
    with np.load(root/'data/followup/B03018A2.npz',allow_pickle=False) as f:data={k:f[k] for k in f.files}
    design=fu_native_design(data['N'],data['counts'],data['blocks']);rows=[];source_support=[]
    for heldout in np.unique(data['component']):
        train=data['component']!=heldout;test=~train
        if not train.any() or not test.any():continue
        d={'Z':design['Z'][train],'T':design['Z'][test],'blocks':data['blocks'][train],'source':design['source']}
        for panel,genes in p['native']['readout_panels'].items():
            gi=[list(data['genes']).index(g) for g in genes];Y=data['Y'][:,gi]
            for model in list(FU_MODELS)+['background_only','train_mean']:
                row={'panel':panel,'heldout_component':int(heldout),'model':model,'training_receivers':int(train.sum()),'test_receivers':int(test.sum())};t=time.perf_counter()
                try:
                    if model=='train_mean':pred=np.broadcast_to(Y[train].mean(0),(test.sum(),4));zero=pred
                    else:
                        fit=fu_fit(d,Y[train],model,bridge);pred=fit['observed_prediction'];zero=fit['observed_zero_prediction']
                        (dest/f'component_parameters_{panel}_{heldout}_{model}.json').write_text(json.dumps(ex_jsonable(fit['audit']),indent=2))
                    row.update(status='ok',source_on=fu_native_metrics(pred,Y[test],data['blocks'][test]),source_zero=fu_native_metrics(zero,Y[test],data['blocks'][test]))
                    np.savez_compressed(dest/f'component_prediction_{panel}_{heldout}_{model}.npz',prediction=pred,zero=zero,Y=Y[test],blocks=data['blocks'][test])
                except Exception as e:row.update(status='failure',error=repr(e))
                row['seconds']=time.perf_counter()-t;rows.append(row);print('COMPONENT',heldout,panel,model,row['status'],flush=True)
        for target in p['native']['source_targets']:
            sm=(data['source_labels']==target)&(data['source_component']==heldout)
            for gi,gene in enumerate(data['genes']):
                source_support.append({'heldout_component':int(heldout),'target':target,'gene':gene,
                    'test_sources':int(sm.sum()),'test_positive_receivers':int((data['Y'][test,gi]>0).sum()),
                    'test_blocks':int(len(np.unique(data['blocks'][test]))),
                    'training_positive_receivers':int((data['Y'][train,gi]>0).sum())})
    with (dest/'components.jsonl').open('w') as f:
        for row in rows:fu_write_row(f,row)
    pd.DataFrame(source_support).to_csv(dest/'component_support.csv',index=False);bridge.close()
    return rows


## 8. Fit the bundled pilot

Run this cell from the repository root. Training uses C02943C3; predictions use only the features from A03599E2. The source contrast keeps the background features fixed. Set `RUN_DEMO = True` to fit.


In [ ]:
from pathlib import Path
ROOT = Path.cwd()
RUN_DEMO = False
if RUN_DEMO:
    chips, outcomes = load_pilot(ROOT)
    design = make_design(chips['C02943C3'], chips['A03599E2'])
    fit = fit_cade(
        design['Z'][:, :5], design['Z'][:, 5:],
        outcomes['C02943C3'], design['blocks'],
        predict_N=design['T'][:, :5], predict_S=design['T'][:, 5:],
    )
    effect = fit['observed_prediction'] - fit['observed_zero_prediction']
    print('Prediction shape:', fit['observed_prediction'].shape)
    print('Source contrast shape:', effect.shape)


## 9. Reproduce experiments

See `README.md` for the command-line tasks. Large experiments write fresh outputs under ignored `results/` or `reruns/` directories and retain their prespecified settings. The package contains executable methods and compact inputs; manuscript drafts, historical reports and prediction caches are not bundled.
